Le notebook doit répondre complètement au premier pilier :

#Predictive Evidence

Question : Does multi-omics integration significantly improve predictive performance?

À la fin de ce notebook, le lecteur doit pouvoir répondre uniquement à cette question.

#1.PCA 50

In [ ]:
import os
import sys
import json
import gzip
import pickle
import hashlib
import platform
import tarfile
import warnings
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Any, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.model_selection import RepeatedStratifiedKFold

import sklearn
import scipy

warnings.filterwarnings("ignore")

# =============================================================================
# 0. CHEMINS, CONFIGURATION, PROVENANCE
# =============================================================================

try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data")
except Exception:
    DATA_DIR = Path("./TCGA_BRCA_data")

RUN_ID = os.environ.get("RUN_ID") or datetime.now().strftime("%Y-%m-%d")
RUN_DIR = DATA_DIR / "runs" / RUN_ID
CKPT_DIR = RUN_DIR / "checkpoints"
OUT_DIR = RUN_DIR / "results"

for d in (DATA_DIR, RUN_DIR, CKPT_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CONFIG: Dict[str, Any] = {
    "cv_folds": 5,
    "cv_repeats": 10,
    "seed": 42,

    # Échantillons TCGA
    "allowed_sample_type_codes": ["01"],
    "duplicate_rule": "first_sorted",

    # mRNA / CNV
    "filter_stat": "mad",
    "filter_top_k": 5000,
    "min_variance": 1e-6,

    # Mutations
    "mut_freq_lo": 0.01,
    "mut_freq_hi": 0.99,

    # RPPA
    "rppa_max_missing": 0.20,
    "rppa_imputer": "knn",  # "knn" ou "median"
    "rppa_knn_k": 5,

    # Réduction dimensionnelle
    "dim_mode": "fixed_k",  # "fixed_k" ou "fixed_variance"
    "dim_k": 50,
    "dim_variance_target": 0.80,
    "dim_k_max": 120,

    # Échelle mRNA — déjà appliquée par UCSC Xena
    "mrna_scale": "log2(TPM + 1)",
    "mrna_apply_log_transform": False,

    # Cache
    "cache_save_every": 5,
}

CONFIG_CANONICAL = deepcopy(CONFIG)
SEED = int(CONFIG["seed"])

LAYERS = ["mutations", "cnv", "mrna", "rppa"]
SHORT = {
    "mutations": "Mutations",
    "cnv": "CNV",
    "mrna": "mRNA",
    "rppa": "RPPA",
}
BINARY_LAYERS = {"mutations"}

NONSYNONYMOUS = {
    "Missense_Mutation",
    "Nonsense_Mutation",
    "Frame_Shift_Del",
    "Frame_Shift_Ins",
    "In_Frame_Del",
    "In_Frame_Ins",
    "Splice_Site",
    "Nonstop_Mutation",
    "Translation_Start_Site",
}

PAM50_GENES = [
    "ACTR3B", "ANLN", "BAG1", "BCL2", "BIRC5", "BLVRA", "CCNB1",
    "CCNE1", "CDC20", "CDC6", "CDH3", "CENPF", "CEP55", "CXXC5",
    "EGFR", "ERBB2", "ESR1", "EXO1", "FGFR4", "FOXA1", "FOXC1",
    "GPR160", "GRB7", "KIF2C", "KRT14", "KRT17", "KRT5", "MAPT",
    "MDM2", "MELK", "MIA", "MKI67", "MLPH", "MMP11", "MYBL2",
    "MYC", "NAT1", "NDC80", "NUF2", "ORC6", "PGR", "PHGDH",
    "PTTG1", "RRM2", "SFRP1", "SLC39A6", "TMEM45B", "TYMS",
    "UBE2C", "UBE2T",
]

ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "scipy": scipy.__version__,
}

LOG: List[str] = []
INPUTS: Dict[str, Any] = {}
VAR_MISS: Dict[str, Dict[str, Any]] = {}
CROSS_LAYER_SELECTIONS: Dict[str, Dict[str, str]] = {}


def log(message: str) -> None:
    print(message)
    LOG.append(f"{datetime.now().strftime('%H:%M:%S')}  {message}")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def register_input(name: str, path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Fichier manquant : {path}")
    INPUTS[name] = {
        "path": str(path),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }


def ckpt_save(name: str, data: Any) -> None:
    target = CKPT_DIR / f"ckpt_{name}.pkl"
    tmp = target.with_suffix(".pkl.tmp")
    payload = {
        "data": data,
        "run": RUN_ID,
        "config": CONFIG_CANONICAL,
        "env": ENV,
        "inputs": deepcopy(INPUTS),
        "written": datetime.now().isoformat(timespec="seconds"),
    }
    with open(tmp, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, target)


def ckpt_load(name: str) -> Optional[Any]:
    path = CKPT_DIR / f"ckpt_{name}.pkl"
    if not path.exists():
        return None

    with open(path, "rb") as f:
        payload = pickle.load(f)

    if payload.get("config") != CONFIG_CANONICAL:
        log(f"[ckpt] {name}: configuration différente, checkpoint ignoré.")
        return None

    if payload.get("inputs") != INPUTS:
        log(f"[ckpt] {name}: fichiers d'entrée différents, checkpoint ignoré.")
        return None

    log(f"[ckpt] {name}: rechargé.")
    return payload["data"]


def assert_finite_matrix(X: np.ndarray, label: str) -> None:
    if X.ndim != 2:
        raise RuntimeError(f"{label}: matrice non bidimensionnelle, shape={X.shape}")
    if X.shape[0] == 0 or X.shape[1] == 0:
        raise RuntimeError(f"{label}: matrice vide, shape={X.shape}")
    if not np.isfinite(X).all():
        n_bad = int((~np.isfinite(X)).sum())
        raise RuntimeError(f"{label}: {n_bad} valeurs NaN ou infinies détectées")


# =============================================================================
# 1. IDENTIFIANTS TCGA ET SÉLECTION DES TUMEURS PRIMAIRES
# =============================================================================

def tcga_patient_id(barcode: str) -> str:
    parts = str(barcode).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(barcode)


def tcga_sample_type_code(barcode: str) -> Optional[str]:
    parts = str(barcode).split("-")
    if len(parts) < 4 or len(parts[3]) < 2:
        return None
    return parts[3][:2]


def tcga_sample_suffix(barcode: Optional[str]) -> Optional[str]:
    if barcode is None:
        return None
    parts = str(barcode).split("-")
    if len(parts) < 4:
        return None
    return parts[3]


def select_primary_tumor_samples(sample_ids: List[str]) -> Dict[str, str]:
    grouped: Dict[str, List[str]] = {}
    allowed = set(CONFIG["allowed_sample_type_codes"])

    for barcode in map(str, sample_ids):
        if tcga_sample_type_code(barcode) not in allowed:
            continue
        grouped.setdefault(tcga_patient_id(barcode), []).append(barcode)

    if CONFIG["duplicate_rule"] != "first_sorted":
        raise ValueError(
            f"Règle de doublon non implémentée : {CONFIG['duplicate_rule']}"
        )

    return {
        patient: sorted(barcodes)[0]
        for patient, barcodes in grouped.items()
    }


def audit_duplicate_primary_samples(
    sample_ids: List[str],
    layer: str,
) -> pd.DataFrame:
    grouped: Dict[str, List[str]] = {}
    allowed = set(CONFIG["allowed_sample_type_codes"])

    for barcode in map(str, sample_ids):
        if tcga_sample_type_code(barcode) not in allowed:
            continue
        grouped.setdefault(tcga_patient_id(barcode), []).append(barcode)

    records = []
    for patient, barcodes in grouped.items():
        if len(barcodes) > 1:
            ordered = sorted(barcodes)
            records.append({
                "layer": layer,
                "patient_id": patient,
                "n_primary_barcodes": len(ordered),
                "barcodes": "|".join(ordered),
                "selected": ordered[0],
            })

    return pd.DataFrame(records)


def collapse_matrix_to_primary_patients(
    matrix: pd.DataFrame,
    samples_are_rows: bool,
    layer_name: str,
) -> pd.DataFrame:
    df = matrix.copy() if samples_are_rows else matrix.T.copy()

    if df.index.duplicated().any():
        duplicated = df.index[df.index.duplicated()].unique().tolist()
        raise RuntimeError(
            f"[{layer_name}] barcodes exactement dupliqués avant sélection : "
            f"{duplicated[:10]}"
        )

    all_ids = df.index.astype(str).tolist()
    n_raw_barcodes = len(all_ids)

    duplicate_audit = audit_duplicate_primary_samples(all_ids, layer_name)
    if not duplicate_audit.empty:
        path = OUT_DIR / f"duplicate_primary_samples_{layer_name}.csv"
        duplicate_audit.to_csv(path, index=False)
        log(
            f"    [{layer_name}] {len(duplicate_audit)} patients avec plusieurs "
            f"barcodes primaires -> {path.name}"
        )

    selected = select_primary_tumor_samples(all_ids)
    if not selected:
        raise ValueError(
            f"[{layer_name}] aucun échantillon tumoral primaire code 01 détecté."
        )

    rows = []
    patient_ids = []
    for patient_id, barcode in selected.items():
        row = df.loc[barcode]
        if isinstance(row, pd.DataFrame):
            raise RuntimeError(
                f"[{layer_name}] df.loc[{barcode!r}] retourne plusieurs lignes."
            )
        rows.append(row)
        patient_ids.append(patient_id)

    out = pd.DataFrame(rows, index=patient_ids)
    out.index.name = "PATIENT_ID"

    CROSS_LAYER_SELECTIONS[layer_name] = selected

    log(
        f"    [{layer_name}] barcodes bruts={n_raw_barcodes} -> "
        f"patients primaires uniques={len(out)}"
    )
    return out.sort_index()


def read_omics_matrix_gz(path: Path, layer_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", index_col=0, compression="gzip")
    return collapse_matrix_to_primary_patients(
        df,
        samples_are_rows=False,
        layer_name=layer_name,
    )


# =============================================================================
# 2. AUDIT DE CONCORDANCE INTER-COUCHES
# =============================================================================

def audit_cross_layer_aliquot_concordance(
    selected_by_layer: Dict[str, Dict[str, str]]
) -> pd.DataFrame:
    all_patients = set()
    for selections in selected_by_layer.values():
        all_patients.update(selections.keys())

    records = []

    for patient_id in sorted(all_patients):
        row: Dict[str, Any] = {"patient_id": patient_id}
        suffixes: Dict[str, str] = {}

        for layer in LAYERS:
            barcode = selected_by_layer.get(layer, {}).get(patient_id)
            suffix = tcga_sample_suffix(barcode)
            row[f"{layer}_barcode"] = barcode
            row[f"{layer}_suffix"] = suffix

            if suffix is not None:
                suffixes[layer] = suffix

        unique_suffixes = set(suffixes.values())
        n_layers_present = len(suffixes)

        row["n_layers_present"] = n_layers_present
        row["evaluable"] = n_layers_present >= 2
        row["concordant"] = (
            n_layers_present >= 2
            and len(unique_suffixes) == 1
        )
        row["four_layer_evaluable"] = n_layers_present == len(LAYERS)
        row["four_layer_concordant"] = (
            n_layers_present == len(LAYERS)
            and len(unique_suffixes) == 1
        )

        records.append(row)

    return pd.DataFrame(records)


# =============================================================================
# 3. mRNA XENA : RECONSTRUCTION, MAPPING HUGO, ÉCHELLE
# =============================================================================

def strip_ensembl_version(gene_id: str) -> str:
    return str(gene_id).split(".")[0]


def audit_mrna_scale(df: pd.DataFrame) -> Dict[str, float]:
    values = df.to_numpy(dtype=np.float64)
    finite = values[np.isfinite(values)]

    if finite.size == 0:
        raise RuntimeError("Aucune valeur mRNA finie détectée.")

    return {
        "min": float(np.min(finite)),
        "median": float(np.median(finite)),
        "p95": float(np.percentile(finite, 95)),
        "p99": float(np.percentile(finite, 99)),
        "max": float(np.max(finite)),
        "fraction_zero": float(np.mean(finite == 0)),
    }


def build_mrna_from_xena(
    xena_path: Path,
    mapping_path: Path,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    """
    Reconstruit la matrice mRNA à partir de la matrice UCSC Xena STAR-TPM.

    IMPORTANT :
    - les valeurs Xena sont déjà sur l'échelle log2(TPM + 1) ;
    - aucune nouvelle transformation logarithmique n'est appliquée.
    """
    if CONFIG["mrna_apply_log_transform"]:
        raise RuntimeError(
            "CONFIG['mrna_apply_log_transform'] doit rester False : "
            "la matrice Xena est déjà en log2(TPM + 1)."
        )

    log("  [mrna] reconstruction depuis la matrice UCSC Xena STAR-TPM...")

    raw = pd.read_csv(
        xena_path,
        sep="\t",
        index_col=0,
        compression="gzip",
    ).T

    primary = collapse_matrix_to_primary_patients(
        raw,
        samples_are_rows=True,
        layer_name="mrna",
    )

    if not mapping_path.exists():
        raise FileNotFoundError(
            f"Mapping Ensembl->HUGO introuvable : {mapping_path}"
        )

    with open(mapping_path, encoding="utf-8") as f:
        id_to_symbol = json.load(f)

    stripped_cols = [
        strip_ensembl_version(column)
        for column in primary.columns
    ]
    mapped_cols = [
        id_to_symbol.get(gene_id)
        for gene_id in stripped_cols
    ]

    n_before = primary.shape[1]
    keep_mask = [symbol is not None for symbol in mapped_cols]

    primary = primary.loc[:, keep_mask]
    primary.columns = [
        symbol for symbol in mapped_cols if symbol is not None
    ]

    n_after_mapping = primary.shape[1]

    # Fusion des identifiants Ensembl mappés au même symbole HUGO
    primary = primary.T.groupby(level=0).mean().T

    log(
        f"    mapping HUGO : {n_before} -> {n_after_mapping} colonnes mappées "
        f"-> {primary.shape[1]} symboles uniques"
    )

    pam50_missing = sorted(set(PAM50_GENES) - set(primary.columns))
    if pam50_missing:
        raise RuntimeError(
            f"Gènes PAM50 manquants après reconstruction mRNA : "
            f"{pam50_missing}"
        )

    log("    contrôle PAM50 : 50/50 gènes présents")

    scale_audit = audit_mrna_scale(primary)
    log(f"    échelle déclarée : {CONFIG['mrna_scale']}")
    log(f"    audit descriptif mRNA : {scale_audit}")

    return primary.sort_index(), scale_audit


# =============================================================================
# 4. FICHIERS D'ENTRÉE
# =============================================================================

MRNA_XENA_PATH = DATA_DIR / "TCGA-BRCA.star_tpm.tsv.gz"
MRNA_REBUILT_PATH = DATA_DIR / "mrna_hugo_mapped_primary_only.parquet"
MRNA_MANIFEST_PATH = (
    DATA_DIR / "mrna_hugo_mapped_primary_only.manifest.json"
)

CNV_PATH = DATA_DIR / "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz"
RPPA_PATH = DATA_DIR / "RPPA_RBN.gz"
ARCHIVE_PATH = DATA_DIR / "brca_tcga_pan_can_atlas_2018.tar.gz"
MAPPING_PATH = DATA_DIR / "ensembl_to_hugo_mapping.json"

for name, path in {
    "mrna_xena_star_tpm": MRNA_XENA_PATH,
    "cnv": CNV_PATH,
    "rppa": RPPA_PATH,
    "mutations_clinical_archive": ARCHIVE_PATH,
    "ensembl_to_hugo_mapping": MAPPING_PATH,
}.items():
    register_input(name, path)

RAW: Dict[str, pd.DataFrame] = {}
MRNA_SCALE_AUDIT: Dict[str, float] = {}

log("=" * 80)
log(f"RUN_ID : {RUN_ID}")
log(f"mRNA scale : {CONFIG['mrna_scale']}")
log("Aucune seconde transformation logarithmique ne sera appliquée.")
log("=" * 80)


def current_mrna_manifest() -> Dict[str, Any]:
    return {
        "xena_sha256": sha256_file(MRNA_XENA_PATH),
        "mapping_sha256": sha256_file(MAPPING_PATH),
        "allowed_sample_type_codes": sorted(
            CONFIG["allowed_sample_type_codes"]
        ),
        "duplicate_rule": CONFIG["duplicate_rule"],
        "mrna_scale": CONFIG["mrna_scale"],
        "mrna_apply_log_transform": CONFIG["mrna_apply_log_transform"],
    }


mrna_manifest_valid = False

if MRNA_REBUILT_PATH.exists() and MRNA_MANIFEST_PATH.exists():
    with open(MRNA_MANIFEST_PATH, encoding="utf-8") as f:
        saved_manifest = json.load(f)

    current_manifest = current_mrna_manifest()
    manifest_keys = [
        "xena_sha256",
        "mapping_sha256",
        "allowed_sample_type_codes",
        "duplicate_rule",
        "mrna_scale",
        "mrna_apply_log_transform",
    ]

    mrna_manifest_valid = all(
        saved_manifest.get(key) == current_manifest.get(key)
        for key in manifest_keys
    )

    if not mrna_manifest_valid:
        log(
            "  [mrna] cache trouvé mais manifeste invalide : "
            "reconstruction forcée."
        )


if MRNA_REBUILT_PATH.exists() and mrna_manifest_valid:
    log(
        f"  [mrna] chargé depuis le cache validé : "
        f"{MRNA_REBUILT_PATH.name}"
    )

    RAW["mrna"] = pd.read_parquet(MRNA_REBUILT_PATH)
    RAW["mrna"].index = RAW["mrna"].index.astype(str)

    pam50_missing = sorted(
        set(PAM50_GENES) - set(RAW["mrna"].columns)
    )
    if pam50_missing:
        raise RuntimeError(
            f"Gènes PAM50 manquants dans le cache mRNA : "
            f"{pam50_missing}"
        )

    log("    contrôle PAM50 du cache : 50/50 gènes présents")

    MRNA_SCALE_AUDIT = audit_mrna_scale(RAW["mrna"])
    log(f"    audit descriptif mRNA : {MRNA_SCALE_AUDIT}")

    with open(MRNA_MANIFEST_PATH, encoding="utf-8") as f:
        saved_manifest_full = json.load(f)

    CROSS_LAYER_SELECTIONS["mrna"] = (
        saved_manifest_full.get("selected_samples", {})
    )

else:
    RAW["mrna"], MRNA_SCALE_AUDIT = build_mrna_from_xena(
        MRNA_XENA_PATH,
        MAPPING_PATH,
    )

    RAW["mrna"].to_parquet(MRNA_REBUILT_PATH)

    manifest = current_mrna_manifest()
    manifest["created_at"] = datetime.now().isoformat(timespec="seconds")
    manifest["scale_audit"] = MRNA_SCALE_AUDIT
    manifest["selected_samples"] = CROSS_LAYER_SELECTIONS.get("mrna", {})

    with open(MRNA_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    log(
        f"  [mrna] cache reconstruit sauvegardé : "
        f"{MRNA_REBUILT_PATH.name}"
    )

register_input(
    "mrna_hugo_mapped_primary_only",
    MRNA_REBUILT_PATH,
)


# =============================================================================
# 5. CHARGEMENT CNV, RPPA, MUTATIONS ET PAM50
# =============================================================================

log("  [cnv] lecture...")
RAW["cnv"] = read_omics_matrix_gz(
    CNV_PATH,
    layer_name="cnv",
)

log("  [rppa] lecture...")
RAW["rppa"] = read_omics_matrix_gz(
    RPPA_PATH,
    layer_name="rppa",
)

log("  [mutations] lecture depuis l'archive...")

with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
    maf_handle = tar.extractfile(
        "brca_tcga_pan_can_atlas_2018/data_mutations.txt"
    )
    clinical_handle = tar.extractfile(
        "brca_tcga_pan_can_atlas_2018/data_clinical_patient.txt"
    )

    if maf_handle is None or clinical_handle is None:
        raise FileNotFoundError(
            "Fichiers mutation/clinique absents de l'archive."
        )

    maf = pd.read_csv(
        maf_handle,
        sep="\t",
        comment="#",
        low_memory=False,
    )

    clinical = pd.read_csv(
        clinical_handle,
        sep="\t",
        comment="#",
        low_memory=False,
    )

required_maf_columns = {
    "Tumor_Sample_Barcode",
    "Hugo_Symbol",
    "Variant_Classification",
}

if not required_maf_columns.issubset(maf.columns):
    raise KeyError(
        f"Colonnes MAF manquantes : "
        f"{required_maf_columns - set(maf.columns)}"
    )

n_maf_before = len(maf)

maf = maf[
    maf["Variant_Classification"].isin(NONSYNONYMOUS)
].copy()

all_mutation_barcodes = (
    maf["Tumor_Sample_Barcode"]
    .astype(str)
    .unique()
    .tolist()
)

duplicate_mutations = audit_duplicate_primary_samples(
    all_mutation_barcodes,
    layer="mutations",
)

if not duplicate_mutations.empty:
    path = OUT_DIR / "duplicate_primary_samples_mutations.csv"
    duplicate_mutations.to_csv(path, index=False)
    log(
        f"    [mutations] {len(duplicate_mutations)} patients avec plusieurs "
        f"barcodes primaires -> {path.name}"
    )

selected_mutation_samples = select_primary_tumor_samples(
    all_mutation_barcodes
)

CROSS_LAYER_SELECTIONS["mutations"] = selected_mutation_samples

selected_mutation_barcodes = set(
    selected_mutation_samples.values()
)

maf = maf[
    maf["Tumor_Sample_Barcode"]
    .astype(str)
    .isin(selected_mutation_barcodes)
].copy()

maf["pid"] = maf["Tumor_Sample_Barcode"].map(
    tcga_patient_id
)

log(
    f"    MAF : {n_maf_before} variantes -> {len(maf)} variantes "
    f"non synonymes, un barcode primaire par patient "
    f"({len(selected_mutation_barcodes)} patients)"
)

RAW["mutations"] = (
    maf.groupby(["pid", "Hugo_Symbol"])
    .size()
    .unstack(fill_value=0)
    .clip(upper=1)
    .astype(np.int8)
    .sort_index()
)

if "SUBTYPE" not in clinical.columns:
    raise KeyError("Colonne SUBTYPE absente du fichier clinique.")

if "PATIENT_ID" not in clinical.columns:
    raise KeyError("Colonne PATIENT_ID absente du fichier clinique.")

labels_all = (
    clinical.set_index("PATIENT_ID")["SUBTYPE"]
    .dropna()
    .astype(str)
    .str.replace(r"^BRCA_", "", regex=True)
)

labels_all = labels_all[
    labels_all.isin(["LumA", "LumB", "Her2", "Basal", "Normal"])
]

labels_all.index = labels_all.index.map(tcga_patient_id)
labels_all.name = "PAM50"

for layer, df in RAW.items():
    if df.index.duplicated().any():
        duplicated = df.index[df.index.duplicated()].unique().tolist()
        raise RuntimeError(
            f"Patients dupliqués dans {layer} : {duplicated[:10]}"
        )


# =============================================================================
# 6. COHORTE FINALE ET AUDIT DE CONCORDANCE
# =============================================================================

common = set(labels_all.index)

for layer in LAYERS:
    common &= set(RAW[layer].index)

common = sorted(common)

if not common:
    raise ValueError(
        "Intersection vide entre les quatre omiques et PAM50."
    )

RAW = {
    layer: RAW[layer].loc[common].copy()
    for layer in LAYERS
}

labels = labels_all.loc[common].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels.to_numpy())

N = len(y)
class_counts = dict(
    zip(
        label_encoder.classes_,
        np.bincount(y).tolist(),
    )
)

log("")
log(f"Cohorte finale : {N} patients")
log(f"Classes PAM50 : {class_counts}")

for layer in LAYERS:
    log(f"{SHORT[layer]:<12}: {RAW[layer].shape}")

if np.bincount(y).min() < int(CONFIG["cv_folds"]):
    raise ValueError(
        "Une classe contient moins de patients que le nombre de folds."
    )

cohort_manifest = pd.DataFrame({
    "patient_id": common,
    "pam50": labels.values,
    "label_encoded": y,
})

cohort_manifest.to_csv(
    OUT_DIR / "cohort_manifest.csv",
    index=False,
)

# Audit de concordance global
if set(CROSS_LAYER_SELECTIONS.keys()) == set(LAYERS):
    concordance_df = audit_cross_layer_aliquot_concordance(
        CROSS_LAYER_SELECTIONS
    )

    concordance_df.to_csv(
        OUT_DIR / "cross_layer_aliquot_concordance_all.csv",
        index=False,
    )

    evaluable_all = concordance_df["evaluable"]
    if evaluable_all.any():
        concordant_all = int(
            concordance_df.loc[evaluable_all, "concordant"].sum()
        )
        n_evaluable_all = int(evaluable_all.sum())

        log(
            f"Concordance inter-couches globale : "
            f"{concordant_all}/{n_evaluable_all} patients évaluables "
            f"({100 * concordant_all / n_evaluable_all:.1f} %)"
        )

    # Audit restreint à la cohorte finale complète
    concordance_complete = concordance_df[
        concordance_df["patient_id"].isin(common)
    ].copy()

    concordance_complete.to_csv(
        OUT_DIR / "cross_layer_aliquot_concordance_complete_cohort.csv",
        index=False,
    )

    n_complete = len(concordance_complete)
    n_four_layer_evaluable = int(
        concordance_complete["four_layer_evaluable"].sum()
    )
    n_four_layer_concordant = int(
        concordance_complete["four_layer_concordant"].sum()
    )

    if n_four_layer_evaluable != n_complete:
        log(
            "ATTENTION : certains patients de la cohorte complète ne sont pas "
            "évaluables sur les quatre couches dans l'audit des aliquotes."
        )

    if n_four_layer_evaluable > 0:
        concordance_rate_complete = (
            100.0
            * n_four_layer_concordant
            / n_four_layer_evaluable
        )
    else:
        concordance_rate_complete = float("nan")

    log(
        f"Concordance des aliquotes dans la cohorte finale : "
        f"{n_four_layer_concordant}/{n_four_layer_evaluable} "
        f"({concordance_rate_complete:.1f} %)"
    )

else:
    missing_selection_layers = sorted(
        set(LAYERS) - set(CROSS_LAYER_SELECTIONS.keys())
    )

    log(
        "Audit de concordance inter-couches impossible. "
        f"Sélections manquantes : {missing_selection_layers}"
    )

    concordance_df = pd.DataFrame()
    concordance_complete = pd.DataFrame()
    n_four_layer_evaluable = 0
    n_four_layer_concordant = 0
    concordance_rate_complete = float("nan")


# =============================================================================
# 7. SPLITS DE VALIDATION CROISÉE PARTAGÉS
# =============================================================================

NF = int(CONFIG["cv_folds"])
NR = int(CONFIG["cv_repeats"])

splitter = RepeatedStratifiedKFold(
    n_splits=NF,
    n_repeats=NR,
    random_state=SEED,
)

SPLITS: List[Tuple[np.ndarray, np.ndarray]] = list(
    splitter.split(np.zeros((N, 1)), y)
)

REPEAT_OF = [
    split_index // NF
    for split_index in range(len(SPLITS))
]

FOLD_OF = [
    split_index % NF
    for split_index in range(len(SPLITS))
]

log(f"{len(SPLITS)} splits = {NR} répétitions x {NF} folds")

with open(
    OUT_DIR / "cv_splits.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        [
            {
                "split": split_index,
                "repeat": REPEAT_OF[split_index],
                "fold": FOLD_OF[split_index],
                "train_indices": train_index.tolist(),
                "test_indices": test_index.tolist(),
            }
            for split_index, (train_index, test_index)
            in enumerate(SPLITS)
        ],
        f,
        indent=2,
    )


# =============================================================================
# 8. PRÉTRAITEMENT INTRA-FOLD
# =============================================================================

def score_features(
    df: pd.DataFrame,
    stat: str,
) -> pd.Series:
    if stat == "mad":
        medians = df.median(axis=0)
        return (df - medians).abs().median(axis=0)

    if stat == "iqr":
        quantiles = df.quantile([0.25, 0.75], axis=0)
        return quantiles.loc[0.75] - quantiles.loc[0.25]

    if stat == "var":
        return df.var(axis=0)

    raise ValueError(f"Statistique de dispersion inconnue : {stat}")


def fit_reducer(
    X_train: np.ndarray,
    layer: str,
    seed: int,
):
    max_allowed = min(
        X_train.shape[0] - 1,
        X_train.shape[1] - 1,
    )

    if max_allowed < 1:
        raise ValueError(
            f"Réduction impossible pour {layer}, shape={X_train.shape}"
        )

    Reducer = (
        TruncatedSVD
        if layer in BINARY_LAYERS
        else PCA
    )

    if CONFIG["dim_mode"] == "fixed_k":
        n_components = min(
            int(CONFIG["dim_k"]),
            max_allowed,
        )

        reducer = Reducer(
            n_components=n_components,
            random_state=seed,
        )

        reducer.fit(X_train)

        retained_variance = float(
            reducer.explained_variance_ratio_.sum()
        )

        return reducer, n_components, retained_variance

    if CONFIG["dim_mode"] == "fixed_variance":
        k_max = min(
            int(CONFIG["dim_k_max"]),
            max_allowed,
        )

        probe = Reducer(
            n_components=k_max,
            random_state=seed,
        )

        probe.fit(X_train)

        cumulative_variance = np.cumsum(
            probe.explained_variance_ratio_
        )

        target = float(CONFIG["dim_variance_target"])

        if cumulative_variance[-1] >= target:
            n_components = int(
                np.searchsorted(
                    cumulative_variance,
                    target,
                )
                + 1
            )
        else:
            n_components = k_max

            miss = VAR_MISS.setdefault(
                layer,
                {
                    "occurrences": 0,
                    "variance_min": 1.0,
                    "variance_max": 0.0,
                    "components_cap": k_max,
                },
            )

            miss["occurrences"] += 1
            miss["variance_min"] = min(
                miss["variance_min"],
                float(cumulative_variance[-1]),
            )
            miss["variance_max"] = max(
                miss["variance_max"],
                float(cumulative_variance[-1]),
            )

        reducer = Reducer(
            n_components=n_components,
            random_state=seed,
        )

        reducer.fit(X_train)

        retained_variance = float(
            reducer.explained_variance_ratio_.sum()
        )

        return reducer, n_components, retained_variance

    raise ValueError(
        f"Mode de réduction inconnu : {CONFIG['dim_mode']}"
    )


def preprocess_layer(
    layer: str,
    train_index: np.ndarray,
    test_index: np.ndarray,
    seed: int,
) -> Dict[str, Any]:
    df = RAW[layer]

    train_df = df.iloc[train_index].copy()
    test_df = df.iloc[test_index].copy()

    metadata: Dict[str, Any] = {
        "layer": layer,
        "n_features_initial": int(train_df.shape[1]),
        "n_train": int(len(train_index)),
        "n_test": int(len(test_index)),
    }

    # -------------------------------------------------------------------------
    # RPPA : missingness appris sur le train + imputation
    # -------------------------------------------------------------------------
    if layer == "rppa":
        missing_rate = train_df.isna().mean(axis=0)

        keep_columns = missing_rate[
            missing_rate <= float(CONFIG["rppa_max_missing"])
        ].index

        train_df = train_df.loc[:, keep_columns]
        test_df = test_df.loc[:, keep_columns]

        metadata["n_features_after_missingness"] = int(
            train_df.shape[1]
        )
        metadata["missing_train_before"] = int(
            train_df.isna().sum().sum()
        )
        metadata["missing_test_before"] = int(
            test_df.isna().sum().sum()
        )

        if CONFIG["rppa_imputer"] == "knn":
            imputer = KNNImputer(
                n_neighbors=int(CONFIG["rppa_knn_k"])
            )
        elif CONFIG["rppa_imputer"] == "median":
            imputer = SimpleImputer(strategy="median")
        else:
            raise ValueError(
                f"Imputeur RPPA inconnu : {CONFIG['rppa_imputer']}"
            )

        n_columns_before = train_df.shape[1]

        train_array = imputer.fit_transform(train_df)
        test_array = imputer.transform(test_df)

        if train_array.shape[1] != n_columns_before:
            raise RuntimeError(
                "[rppa] l'imputation a modifié le nombre de variables : "
                f"{n_columns_before} -> {train_array.shape[1]}"
            )

        train_df = pd.DataFrame(
            train_array,
            index=train_df.index,
            columns=train_df.columns,
        )

        test_df = pd.DataFrame(
            test_array,
            index=test_df.index,
            columns=test_df.columns,
        )

    # -------------------------------------------------------------------------
    # Autres couches : médiane si valeurs manquantes
    # -------------------------------------------------------------------------
    elif (
        train_df.isna().any().any()
        or test_df.isna().any().any()
    ):
        imputer = SimpleImputer(strategy="median")

        train_df = pd.DataFrame(
            imputer.fit_transform(train_df),
            index=train_df.index,
            columns=train_df.columns,
        )

        test_df = pd.DataFrame(
            imputer.transform(test_df),
            index=test_df.index,
            columns=test_df.columns,
        )

    # -------------------------------------------------------------------------
    # Mutations : filtre de fréquence appris sur le train
    # -------------------------------------------------------------------------
    if layer == "mutations":
        mutation_frequency = train_df.mean(axis=0)

        keep_columns = mutation_frequency[
            (
                mutation_frequency
                >= float(CONFIG["mut_freq_lo"])
            )
            & (
                mutation_frequency
                <= float(CONFIG["mut_freq_hi"])
            )
        ].index

        train_df = train_df.loc[:, keep_columns]
        test_df = test_df.loc[:, keep_columns]

        metadata["n_features_after_frequency"] = int(
            train_df.shape[1]
        )

    # -------------------------------------------------------------------------
    # Filtre de variance quasi nulle
    # -------------------------------------------------------------------------
    variances = train_df.var(axis=0)

    keep_columns = variances[
        variances > float(CONFIG["min_variance"])
    ].index

    train_df = train_df.loc[:, keep_columns]
    test_df = test_df.loc[:, keep_columns]

    metadata["n_features_after_variance"] = int(
        train_df.shape[1]
    )

    # -------------------------------------------------------------------------
    # Top-k non supervisé mRNA/CNV
    # -------------------------------------------------------------------------
    if (
        layer in {"mrna", "cnv"}
        and train_df.shape[1] > int(CONFIG["filter_top_k"])
    ):
        feature_scores = score_features(
            train_df,
            str(CONFIG["filter_stat"]),
        )

        selected_columns = feature_scores.nlargest(
            int(CONFIG["filter_top_k"])
        ).index

        train_df = train_df.loc[:, selected_columns]
        test_df = test_df.loc[:, selected_columns]

    if train_df.shape[1] < 2:
        raise ValueError(
            f"{layer}: moins de deux variables après filtrage."
        )

    metadata["n_features_selected"] = int(
        train_df.shape[1]
    )

    metadata["selected_features"] = (
        train_df.columns.astype(str).tolist()
    )

    X_train = train_df.to_numpy(
        dtype=np.float64,
        copy=True,
    )

    X_test = test_df.to_numpy(
        dtype=np.float64,
        copy=True,
    )

    assert_finite_matrix(
        X_train,
        f"{layer}/train avant standardisation-réduction",
    )

    assert_finite_matrix(
        X_test,
        f"{layer}/test avant standardisation-réduction",
    )

    # -------------------------------------------------------------------------
    # Standardisation couches continues
    # -------------------------------------------------------------------------
    if layer not in BINARY_LAYERS:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        metadata["standardized"] = True
    else:
        metadata["standardized"] = False

    assert_finite_matrix(
        X_train,
        f"{layer}/train après standardisation",
    )

    assert_finite_matrix(
        X_test,
        f"{layer}/test après standardisation",
    )

    # -------------------------------------------------------------------------
    # PCA / TruncatedSVD
    # -------------------------------------------------------------------------
    reducer, n_components, retained_variance = fit_reducer(
        X_train,
        layer=layer,
        seed=seed,
    )

    Z_train = reducer.transform(X_train).astype(np.float32)
    Z_test = reducer.transform(X_test).astype(np.float32)

    assert_finite_matrix(
        Z_train,
        f"{layer}/train après réduction",
    )

    assert_finite_matrix(
        Z_test,
        f"{layer}/test après réduction",
    )

    metadata["n_components"] = int(n_components)
    metadata["retained_variance"] = float(retained_variance)
    metadata["dim_mode"] = CONFIG["dim_mode"]
    metadata["filter_stat"] = CONFIG["filter_stat"]

    return {
        "X_train": Z_train,
        "X_test": Z_test,
        "metadata": metadata,
    }


# =============================================================================
# 9. CONSTRUCTION DU CACHE
# =============================================================================

cache = ckpt_load("preprocessing_cache")

if cache is None:
    cache = {
        "representations": {},
        "metadata": {},
    }

REPRESENTATIONS = cache["representations"]
PREPROCESS_METADATA = cache["metadata"]

for split_index, (train_index, test_index) in enumerate(SPLITS):
    split_complete = all(
        (split_index, layer) in REPRESENTATIONS
        for layer in LAYERS
    )

    if split_complete:
        continue

    log(f"Split {split_index + 1}/{len(SPLITS)}")

    for layer in LAYERS:
        result = preprocess_layer(
            layer=layer,
            train_index=train_index,
            test_index=test_index,
            seed=SEED + REPEAT_OF[split_index],
        )

        REPRESENTATIONS[(split_index, layer)] = (
            result["X_train"],
            result["X_test"],
        )

        PREPROCESS_METADATA[(split_index, layer)] = (
            result["metadata"]
        )

    if (
        (split_index + 1)
        % int(CONFIG["cache_save_every"])
        == 0
        or split_index == len(SPLITS) - 1
    ):
        ckpt_save(
            "preprocessing_cache",
            {
                "representations": REPRESENTATIONS,
                "metadata": PREPROCESS_METADATA,
            },
        )


# =============================================================================
# 10. EXPORTS ET RAPPORT FINAL
# =============================================================================

summary_rows = []

for layer in LAYERS:
    rows = [
        PREPROCESS_METADATA[(split_index, layer)]
        for split_index in range(len(SPLITS))
    ]

    n_features = [
        row["n_features_selected"]
        for row in rows
    ]

    n_components = [
        row["n_components"]
        for row in rows
    ]

    retained_variance = [
        row["retained_variance"]
        for row in rows
    ]

    summary_row = {
        "layer": layer,
        "features_median": int(np.median(n_features)),
        "features_min": int(np.min(n_features)),
        "features_max": int(np.max(n_features)),
        "components_median": int(np.median(n_components)),
        "components_min": int(np.min(n_components)),
        "components_max": int(np.max(n_components)),
        "retained_variance_mean": float(
            np.mean(retained_variance)
        ),
        "retained_variance_sd": float(
            np.std(retained_variance, ddof=1)
        ),
    }

    summary_rows.append(summary_row)

    log(
        f"{SHORT[layer]:<12} "
        f"features={summary_row['features_median']} "
        f"components={summary_row['components_median']} "
        f"variance="
        f"{100 * summary_row['retained_variance_mean']:.1f}%"
    )

pd.DataFrame(summary_rows).to_csv(
    OUT_DIR / "representation_summary.csv",
    index=False,
)

flat_metadata = []

for (split_index, layer), metadata in PREPROCESS_METADATA.items():
    flat_metadata.append({
        "split": int(split_index),
        "repeat": int(REPEAT_OF[split_index]),
        "fold": int(FOLD_OF[split_index]),
        **{
            key: value
            for key, value in metadata.items()
            if key != "selected_features"
        },
    })

pd.DataFrame(flat_metadata).to_csv(
    OUT_DIR / "preprocessing_metadata.csv",
    index=False,
)

selected_features_payload = {
    f"split_{split_index}__{layer}":
        PREPROCESS_METADATA[
            (split_index, layer)
        ]["selected_features"]
    for split_index in range(len(SPLITS))
    for layer in LAYERS
}

with gzip.open(
    OUT_DIR / "selected_features_by_fold.json.gz",
    "wt",
    encoding="utf-8",
) as f:
    json.dump(selected_features_payload, f)

report = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "config": CONFIG_CANONICAL,
    "environment": ENV,
    "inputs": INPUTS,

    "mrna": {
        "source": "UCSC Xena GDC STAR-TPM matrix",
        "declared_scale": CONFIG["mrna_scale"],
        "log_transform_applied_in_this_script": False,
        "scale_audit": MRNA_SCALE_AUDIT,
        "pam50_genes_present": 50,
    },

    "cohort": {
        "n_patients": int(N),
        "classes": class_counts,
        "layers": {
            layer: {
                "n_patients": int(RAW[layer].shape[0]),
                "n_features": int(RAW[layer].shape[1]),
            }
            for layer in LAYERS
        },
    },

    "cross_layer_aliquot_concordance_complete_cohort": {
        "n_patients": int(len(common)),
        "n_four_layer_evaluable": int(n_four_layer_evaluable),
        "n_four_layer_concordant": int(n_four_layer_concordant),
        "concordance_rate_percent": (
            None
            if not np.isfinite(concordance_rate_complete)
            else float(concordance_rate_complete)
        ),
    },

    "cross_validation": {
        "folds": NF,
        "repeats": NR,
        "n_splits": len(SPLITS),
    },

    "representation_summary": summary_rows,
    "variance_target_misses": VAR_MISS,

    "design_notes": [
        "Only TCGA primary solid tumor samples (sample type code 01) are retained for all four omics layers.",
        "One primary barcode per patient and per layer is selected deterministically using the alphabetically first barcode.",
        "Cross-layer aliquot concordance is audited rather than enforced, because identical sample suffixes across platforms are not guaranteed.",
        "The mRNA source is a UCSC Xena STAR-TPM matrix already expressed as log2(TPM + 1); no additional logarithmic transformation is applied.",
        "Presence of all 50 PAM50 genes is asserted after mRNA reconstruction and when reloading the mRNA cache.",
        "All data-dependent preprocessing is fitted exclusively on training rows.",
        "The same layer representation is reused unchanged across all single- and multi-omics combinations within each split.",
        "The primary analysis uses fixed-k=50 components per layer.",
        "A fixed-variance representation is reserved for sensitivity analysis.",
        "mRNA and CNV filtering is unsupervised and based on training-fold MAD.",
        "PAM50 labels are not used for feature selection or dimensionality reduction.",
        "Checkpoints are invalidated if either the configuration or any input file SHA-256 changes.",
        "Defensive checks reject exact barcode duplicates and non-finite matrices.",
    ],
}

with open(
    OUT_DIR / "preprocessing_report.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False,
    )

with open(
    OUT_DIR / "preprocessing_run.log",
    "w",
    encoding="utf-8",
) as f:
    f.write("\n".join(LOG))

log("")
log("Prétraitement terminé avec succès.")
log(f"Résultats : {OUT_DIR}")

#1. PCA 100

In [ ]:
"""
===============================================================================
(variante k=100, simple)
===============================================================================

"""

import os
import sys
import json
import gzip
import pickle
import hashlib
import platform
import tarfile
import warnings
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Any, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.model_selection import RepeatedStratifiedKFold

import sklearn
import scipy

warnings.filterwarnings("ignore")

# =============================================================================
# 0. CHEMINS, CONFIGURATION -- RUN_ID FIXÉ À "k100"
# =============================================================================

try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data")
except Exception:
    DATA_DIR = Path("./TCGA_BRCA_data")

RUN_ID = "k100"   # <-- fixé en dur, jamais la date du jour, jamais "k50"
RUN_DIR = DATA_DIR / "runs" / RUN_ID
CKPT_DIR = RUN_DIR / "checkpoints"
OUT_DIR = RUN_DIR / "results"

for d in (DATA_DIR, RUN_DIR, CKPT_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CONFIG: Dict[str, Any] = {
    "cv_folds": 5,
    "cv_repeats": 10,
    "seed": 42,
    "allowed_sample_type_codes": ["01"],
    "duplicate_rule": "first_sorted",
    "filter_stat": "mad",
    "filter_top_k": 5000,
    "min_variance": 1e-6,
    "mut_freq_lo": 0.01,
    "mut_freq_hi": 0.99,
    "rppa_max_missing": 0.20,
    "rppa_imputer": "knn",
    "rppa_knn_k": 5,
    "dim_mode": "fixed_k",
    "dim_k": 100,          # <-- SEUL CHANGEMENT DE FOND : 50 -> 100
    "dim_variance_target": 0.80,
    "dim_k_max": 120,
    "mrna_scale": "log2(TPM + 1)",
    "mrna_apply_log_transform": False,
    "cache_save_every": 5,
}

CONFIG_CANONICAL = deepcopy(CONFIG)
SEED = int(CONFIG["seed"])

LAYERS = ["mutations", "cnv", "mrna", "rppa"]
SHORT = {"mutations": "Mutations", "cnv": "CNV", "mrna": "mRNA", "rppa": "RPPA"}
BINARY_LAYERS = {"mutations"}

NONSYNONYMOUS = {
    "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
    "Frame_Shift_Ins", "In_Frame_Del", "In_Frame_Ins", "Splice_Site",
    "Nonstop_Mutation", "Translation_Start_Site",
}

PAM50_GENES = [
    "ACTR3B", "ANLN", "BAG1", "BCL2", "BIRC5", "BLVRA", "CCNB1",
    "CCNE1", "CDC20", "CDC6", "CDH3", "CENPF", "CEP55", "CXXC5",
    "EGFR", "ERBB2", "ESR1", "EXO1", "FGFR4", "FOXA1", "FOXC1",
    "GPR160", "GRB7", "KIF2C", "KRT14", "KRT17", "KRT5", "MAPT",
    "MDM2", "MELK", "MIA", "MKI67", "MLPH", "MMP11", "MYBL2",
    "MYC", "NAT1", "NDC80", "NUF2", "ORC6", "PGR", "PHGDH",
    "PTTG1", "RRM2", "SFRP1", "SLC39A6", "TMEM45B", "TYMS",
    "UBE2C", "UBE2T",
]

ENV = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__,
    "sklearn": sklearn.__version__, "scipy": scipy.__version__,
}

LOG: List[str] = []
INPUTS: Dict[str, Any] = {}
VAR_MISS: Dict[str, Dict[str, Any]] = {}
CROSS_LAYER_SELECTIONS: Dict[str, Dict[str, str]] = {}


def log(message: str) -> None:
    print(message)
    LOG.append(f"{datetime.now().strftime('%H:%M:%S')}  {message}")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def register_input(name: str, path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Fichier manquant : {path}")
    INPUTS[name] = {
        "path": str(path), "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }


def ckpt_save(name: str, data: Any) -> None:
    target = CKPT_DIR / f"ckpt_{name}.pkl"
    tmp = target.with_suffix(".pkl.tmp")
    payload = {
        "data": data, "run": RUN_ID, "config": CONFIG_CANONICAL,
        "env": ENV, "inputs": deepcopy(INPUTS),
        "written": datetime.now().isoformat(timespec="seconds"),
    }
    with open(tmp, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, target)


def ckpt_load(name: str) -> Optional[Any]:
    path = CKPT_DIR / f"ckpt_{name}.pkl"
    if not path.exists():
        return None
    with open(path, "rb") as f:
        payload = pickle.load(f)
    if payload.get("config") != CONFIG_CANONICAL:
        log(f"[ckpt] {name}: configuration différente, checkpoint ignoré.")
        return None
    if payload.get("inputs") != INPUTS:
        log(f"[ckpt] {name}: fichiers d'entrée différents, checkpoint ignoré.")
        return None
    log(f"[ckpt] {name}: rechargé.")
    return payload["data"]


def assert_finite_matrix(X: np.ndarray, label: str) -> None:
    if X.ndim != 2:
        raise RuntimeError(f"{label}: matrice non bidimensionnelle, shape={X.shape}")
    if X.shape[0] == 0 or X.shape[1] == 0:
        raise RuntimeError(f"{label}: matrice vide, shape={X.shape}")
    if not np.isfinite(X).all():
        n_bad = int((~np.isfinite(X)).sum())
        raise RuntimeError(f"{label}: {n_bad} valeurs NaN ou infinies détectées")


# =============================================================================
# 1. IDENTIFIANTS TCGA ET SÉLECTION DES TUMEURS PRIMAIRES
# =============================================================================

def tcga_patient_id(barcode: str) -> str:
    parts = str(barcode).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(barcode)


def tcga_sample_type_code(barcode: str) -> Optional[str]:
    parts = str(barcode).split("-")
    if len(parts) < 4 or len(parts[3]) < 2:
        return None
    return parts[3][:2]


def tcga_sample_suffix(barcode: Optional[str]) -> Optional[str]:
    if barcode is None:
        return None
    parts = str(barcode).split("-")
    if len(parts) < 4:
        return None
    return parts[3]


def select_primary_tumor_samples(sample_ids: List[str]) -> Dict[str, str]:
    grouped: Dict[str, List[str]] = {}
    allowed = set(CONFIG["allowed_sample_type_codes"])
    for barcode in map(str, sample_ids):
        if tcga_sample_type_code(barcode) not in allowed:
            continue
        grouped.setdefault(tcga_patient_id(barcode), []).append(barcode)
    if CONFIG["duplicate_rule"] != "first_sorted":
        raise ValueError(f"Règle de doublon non implémentée : {CONFIG['duplicate_rule']}")
    return {patient: sorted(barcodes)[0] for patient, barcodes in grouped.items()}


def audit_duplicate_primary_samples(sample_ids: List[str], layer: str) -> pd.DataFrame:
    grouped: Dict[str, List[str]] = {}
    allowed = set(CONFIG["allowed_sample_type_codes"])
    for barcode in map(str, sample_ids):
        if tcga_sample_type_code(barcode) not in allowed:
            continue
        grouped.setdefault(tcga_patient_id(barcode), []).append(barcode)
    records = []
    for patient, barcodes in grouped.items():
        if len(barcodes) > 1:
            ordered = sorted(barcodes)
            records.append({
                "layer": layer, "patient_id": patient,
                "n_primary_barcodes": len(ordered),
                "barcodes": "|".join(ordered), "selected": ordered[0],
            })
    return pd.DataFrame(records)


def collapse_matrix_to_primary_patients(matrix, samples_are_rows, layer_name):
    df = matrix.copy() if samples_are_rows else matrix.T.copy()
    if df.index.duplicated().any():
        duplicated = df.index[df.index.duplicated()].unique().tolist()
        raise RuntimeError(
            f"[{layer_name}] barcodes exactement dupliqués avant sélection : {duplicated[:10]}"
        )
    all_ids = df.index.astype(str).tolist()
    n_raw_barcodes = len(all_ids)
    duplicate_audit = audit_duplicate_primary_samples(all_ids, layer_name)
    if not duplicate_audit.empty:
        path = OUT_DIR / f"duplicate_primary_samples_{layer_name}.csv"
        duplicate_audit.to_csv(path, index=False)
        log(f"    [{layer_name}] {len(duplicate_audit)} patients avec plusieurs "
            f"barcodes primaires -> {path.name}")
    selected = select_primary_tumor_samples(all_ids)
    if not selected:
        raise ValueError(f"[{layer_name}] aucun échantillon tumoral primaire code 01 détecté.")
    rows, patient_ids = [], []
    for patient_id, barcode in selected.items():
        row = df.loc[barcode]
        if isinstance(row, pd.DataFrame):
            raise RuntimeError(f"[{layer_name}] df.loc[{barcode!r}] retourne plusieurs lignes.")
        rows.append(row)
        patient_ids.append(patient_id)
    out = pd.DataFrame(rows, index=patient_ids)
    out.index.name = "PATIENT_ID"
    CROSS_LAYER_SELECTIONS[layer_name] = selected
    log(f"    [{layer_name}] barcodes bruts={n_raw_barcodes} -> "
        f"patients primaires uniques={len(out)}")
    return out.sort_index()


def read_omics_matrix_gz(path: Path, layer_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", index_col=0, compression="gzip")
    return collapse_matrix_to_primary_patients(df, samples_are_rows=False, layer_name=layer_name)


# =============================================================================
# 2. AUDIT DE CONCORDANCE INTER-COUCHES (version simple : un seul niveau)
# =============================================================================

def audit_cross_layer_aliquot_concordance(selected_by_layer):
    all_patients = set()
    for selections in selected_by_layer.values():
        all_patients.update(selections.keys())
    records = []
    for patient_id in sorted(all_patients):
        row = {"patient_id": patient_id}
        suffixes = {}
        for layer in LAYERS:
            barcode = selected_by_layer.get(layer, {}).get(patient_id)
            suffix = tcga_sample_suffix(barcode)
            row[f"{layer}_suffix"] = suffix
            if suffix is not None:
                suffixes[layer] = suffix
        unique_suffixes = set(suffixes.values())
        n_layers_present = len(suffixes)
        row["n_layers_present"] = n_layers_present
        row["evaluable"] = n_layers_present >= 2
        row["concordant"] = n_layers_present >= 2 and len(unique_suffixes) == 1
        row["four_layer_evaluable"] = n_layers_present == len(LAYERS)
        row["four_layer_concordant"] = (
            n_layers_present == len(LAYERS) and len(unique_suffixes) == 1
        )
        records.append(row)
    return pd.DataFrame(records)


# =============================================================================
# 3. mRNA XENA
# =============================================================================

def strip_ensembl_version(gene_id: str) -> str:
    return str(gene_id).split(".")[0]


def audit_mrna_scale(df: pd.DataFrame) -> Dict[str, float]:
    values = df.to_numpy(dtype=np.float64)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        raise RuntimeError("Aucune valeur mRNA finie détectée.")
    return {
        "min": float(np.min(finite)), "median": float(np.median(finite)),
        "p95": float(np.percentile(finite, 95)), "p99": float(np.percentile(finite, 99)),
        "max": float(np.max(finite)), "fraction_zero": float(np.mean(finite == 0)),
    }


def build_mrna_from_xena(xena_path: Path, mapping_path: Path):
    if CONFIG["mrna_apply_log_transform"]:
        raise RuntimeError(
            "CONFIG['mrna_apply_log_transform'] doit rester False : "
            "la matrice Xena est déjà en log2(TPM + 1)."
        )
    log("  [mrna] reconstruction depuis la matrice UCSC Xena STAR-TPM...")
    raw = pd.read_csv(xena_path, sep="\t", index_col=0, compression="gzip").T
    primary = collapse_matrix_to_primary_patients(raw, samples_are_rows=True, layer_name="mrna")
    if not mapping_path.exists():
        raise FileNotFoundError(f"Mapping Ensembl->HUGO introuvable : {mapping_path}")
    with open(mapping_path, encoding="utf-8") as f:
        id_to_symbol = json.load(f)
    stripped_cols = [strip_ensembl_version(c) for c in primary.columns]
    mapped_cols = [id_to_symbol.get(g) for g in stripped_cols]
    n_before = primary.shape[1]
    keep_mask = [s is not None for s in mapped_cols]
    primary = primary.loc[:, keep_mask]
    primary.columns = [s for s in mapped_cols if s is not None]
    n_after_mapping = primary.shape[1]
    primary = primary.T.groupby(level=0).mean().T
    log(f"    mapping HUGO : {n_before} -> {n_after_mapping} colonnes mappées "
        f"-> {primary.shape[1]} symboles uniques")
    pam50_missing = sorted(set(PAM50_GENES) - set(primary.columns))
    if pam50_missing:
        raise RuntimeError(f"Gènes PAM50 manquants après reconstruction mRNA : {pam50_missing}")
    log("    contrôle PAM50 : 50/50 gènes présents")
    scale_audit = audit_mrna_scale(primary)
    log(f"    échelle déclarée : {CONFIG['mrna_scale']}")
    log(f"    audit descriptif mRNA : {scale_audit}")
    return primary.sort_index(), scale_audit


# =============================================================================
# 4. FICHIERS D'ENTRÉE
# =============================================================================

MRNA_XENA_PATH = DATA_DIR / "TCGA-BRCA.star_tpm.tsv.gz"
MRNA_REBUILT_PATH = DATA_DIR / "mrna_hugo_mapped_primary_only.parquet"
MRNA_MANIFEST_PATH = DATA_DIR / "mrna_hugo_mapped_primary_only.manifest.json"
CNV_PATH = DATA_DIR / "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz"
RPPA_PATH = DATA_DIR / "RPPA_RBN.gz"
ARCHIVE_PATH = DATA_DIR / "brca_tcga_pan_can_atlas_2018.tar.gz"
MAPPING_PATH = DATA_DIR / "ensembl_to_hugo_mapping.json"

for name, path in {
    "mrna_xena_star_tpm": MRNA_XENA_PATH, "cnv": CNV_PATH, "rppa": RPPA_PATH,
    "mutations_clinical_archive": ARCHIVE_PATH, "ensembl_to_hugo_mapping": MAPPING_PATH,
}.items():
    register_input(name, path)

RAW: Dict[str, pd.DataFrame] = {}
MRNA_SCALE_AUDIT: Dict[str, float] = {}

log("=" * 80)
log(f"RUN_ID : {RUN_ID}  (dossier : {RUN_DIR})")
log(f"dim_mode : {CONFIG['dim_mode']}  |  dim_k : {CONFIG['dim_k']}")
log("=" * 80)


def current_mrna_manifest() -> Dict[str, Any]:
    return {
        "xena_sha256": sha256_file(MRNA_XENA_PATH),
        "mapping_sha256": sha256_file(MAPPING_PATH),
        "allowed_sample_type_codes": sorted(CONFIG["allowed_sample_type_codes"]),
        "duplicate_rule": CONFIG["duplicate_rule"],
        "mrna_scale": CONFIG["mrna_scale"],
        "mrna_apply_log_transform": CONFIG["mrna_apply_log_transform"],
    }


mrna_manifest_valid = False
if MRNA_REBUILT_PATH.exists() and MRNA_MANIFEST_PATH.exists():
    with open(MRNA_MANIFEST_PATH, encoding="utf-8") as f:
        saved_manifest = json.load(f)
    current_manifest = current_mrna_manifest()
    keys = ["xena_sha256", "mapping_sha256", "allowed_sample_type_codes",
            "duplicate_rule", "mrna_scale", "mrna_apply_log_transform"]
    mrna_manifest_valid = all(saved_manifest.get(k) == current_manifest.get(k) for k in keys)
    if not mrna_manifest_valid:
        log("  [mrna] cache trouvé mais manifeste invalide : reconstruction forcée.")

if MRNA_REBUILT_PATH.exists() and mrna_manifest_valid:
    log(f"  [mrna] chargé depuis le cache validé (partagé avec k50) : {MRNA_REBUILT_PATH.name}")
    RAW["mrna"] = pd.read_parquet(MRNA_REBUILT_PATH)
    RAW["mrna"].index = RAW["mrna"].index.astype(str)
    pam50_missing = sorted(set(PAM50_GENES) - set(RAW["mrna"].columns))
    if pam50_missing:
        raise RuntimeError(f"Gènes PAM50 manquants dans le cache mRNA : {pam50_missing}")
    log("    contrôle PAM50 du cache : 50/50 gènes présents")
    MRNA_SCALE_AUDIT = audit_mrna_scale(RAW["mrna"])
    log(f"    audit descriptif mRNA : {MRNA_SCALE_AUDIT}")
    with open(MRNA_MANIFEST_PATH, encoding="utf-8") as f:
        saved_manifest_full = json.load(f)
    CROSS_LAYER_SELECTIONS["mrna"] = saved_manifest_full.get("selected_samples", {})
else:
    RAW["mrna"], MRNA_SCALE_AUDIT = build_mrna_from_xena(MRNA_XENA_PATH, MAPPING_PATH)
    RAW["mrna"].to_parquet(MRNA_REBUILT_PATH)
    manifest = current_mrna_manifest()
    manifest["created_at"] = datetime.now().isoformat(timespec="seconds")
    manifest["scale_audit"] = MRNA_SCALE_AUDIT
    manifest["selected_samples"] = CROSS_LAYER_SELECTIONS.get("mrna", {})
    with open(MRNA_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    log(f"  [mrna] cache reconstruit sauvegardé : {MRNA_REBUILT_PATH.name}")

register_input("mrna_hugo_mapped_primary_only", MRNA_REBUILT_PATH)

log("  [cnv] lecture...")
RAW["cnv"] = read_omics_matrix_gz(CNV_PATH, layer_name="cnv")
log("  [rppa] lecture...")
RAW["rppa"] = read_omics_matrix_gz(RPPA_PATH, layer_name="rppa")

log("  [mutations] lecture depuis l'archive...")
with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
    maf_handle = tar.extractfile("brca_tcga_pan_can_atlas_2018/data_mutations.txt")
    clinical_handle = tar.extractfile("brca_tcga_pan_can_atlas_2018/data_clinical_patient.txt")
    if maf_handle is None or clinical_handle is None:
        raise FileNotFoundError("Fichiers mutation/clinique absents de l'archive.")
    maf = pd.read_csv(maf_handle, sep="\t", comment="#", low_memory=False)
    clinical = pd.read_csv(clinical_handle, sep="\t", comment="#", low_memory=False)

required_maf_columns = {"Tumor_Sample_Barcode", "Hugo_Symbol", "Variant_Classification"}
if not required_maf_columns.issubset(maf.columns):
    raise KeyError(f"Colonnes MAF manquantes : {required_maf_columns - set(maf.columns)}")

n_maf_before = len(maf)
maf = maf[maf["Variant_Classification"].isin(NONSYNONYMOUS)].copy()
all_mutation_barcodes = maf["Tumor_Sample_Barcode"].astype(str).unique().tolist()
duplicate_mutations = audit_duplicate_primary_samples(all_mutation_barcodes, layer="mutations")
if not duplicate_mutations.empty:
    path = OUT_DIR / "duplicate_primary_samples_mutations.csv"
    duplicate_mutations.to_csv(path, index=False)
    log(f"    [mutations] {len(duplicate_mutations)} patients avec plusieurs "
        f"barcodes primaires -> {path.name}")
selected_mutation_samples = select_primary_tumor_samples(all_mutation_barcodes)
CROSS_LAYER_SELECTIONS["mutations"] = selected_mutation_samples
selected_mutation_barcodes = set(selected_mutation_samples.values())
maf = maf[maf["Tumor_Sample_Barcode"].astype(str).isin(selected_mutation_barcodes)].copy()
maf["pid"] = maf["Tumor_Sample_Barcode"].map(tcga_patient_id)
log(f"    MAF : {n_maf_before} variantes -> {len(maf)} variantes non synonymes, "
    f"un barcode primaire par patient ({len(selected_mutation_barcodes)} patients)")

RAW["mutations"] = (
    maf.groupby(["pid", "Hugo_Symbol"]).size().unstack(fill_value=0)
    .clip(upper=1).astype(np.int8).sort_index()
)

if "SUBTYPE" not in clinical.columns:
    raise KeyError("Colonne SUBTYPE absente du fichier clinique.")
if "PATIENT_ID" not in clinical.columns:
    raise KeyError("Colonne PATIENT_ID absente du fichier clinique.")

labels_all = (
    clinical.set_index("PATIENT_ID")["SUBTYPE"].dropna().astype(str)
    .str.replace(r"^BRCA_", "", regex=True)
)
labels_all = labels_all[labels_all.isin(["LumA", "LumB", "Her2", "Basal", "Normal"])]
labels_all.index = labels_all.index.map(tcga_patient_id)
labels_all.name = "PAM50"

if labels_all.index.duplicated().any():
    duplicated_labels = labels_all.index[labels_all.index.duplicated(keep=False)].unique().tolist()
    consistency = labels_all.groupby(level=0).nunique()
    conflicting = consistency[consistency > 1].index.tolist()
    if conflicting:
        raise RuntimeError(f"Labels PAM50 contradictoires pour certains patients : {conflicting[:10]}")
    labels_all = labels_all[~labels_all.index.duplicated(keep="first")]
    log(f"Labels cliniques dupliqués mais cohérents : {len(duplicated_labels)} patients dédupliqués.")

for layer, df in RAW.items():
    if df.index.duplicated().any():
        duplicated = df.index[df.index.duplicated()].unique().tolist()
        raise RuntimeError(f"Patients dupliqués dans {layer} : {duplicated[:10]}")


# =============================================================================
# 5. COHORTE FINALE ET AUDIT DE CONCORDANCE
# =============================================================================

common = set(labels_all.index)
for layer in LAYERS:
    common &= set(RAW[layer].index)
common = sorted(common)
if not common:
    raise ValueError("Intersection vide entre les quatre omiques et PAM50.")

RAW = {layer: RAW[layer].loc[common].copy() for layer in LAYERS}
labels = labels_all.loc[common].copy()
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels.to_numpy())
N = len(y)
class_counts = dict(zip(label_encoder.classes_, np.bincount(y).tolist()))

log("")
log(f"Cohorte finale : {N} patients")
log(f"Classes PAM50 : {class_counts}")
for layer in LAYERS:
    log(f"{SHORT[layer]:<12}: {RAW[layer].shape}")
if np.bincount(y).min() < int(CONFIG["cv_folds"]):
    raise ValueError("Une classe contient moins de patients que le nombre de folds.")

pd.DataFrame({
    "patient_id": common, "pam50": labels.values, "label_encoded": y,
}).to_csv(OUT_DIR / "cohort_manifest.csv", index=False)

if set(CROSS_LAYER_SELECTIONS.keys()) == set(LAYERS):
    concordance_df = audit_cross_layer_aliquot_concordance(CROSS_LAYER_SELECTIONS)
    concordance_df.to_csv(OUT_DIR / "cross_layer_aliquot_concordance_all.csv", index=False)

    evaluable_all = concordance_df["evaluable"]
    if evaluable_all.any():
        concordant_all = int(concordance_df.loc[evaluable_all, "concordant"].sum())
        n_evaluable_all = int(evaluable_all.sum())
        log(f"Concordance inter-couches globale : {concordant_all}/{n_evaluable_all} "
            f"patients évaluables ({100 * concordant_all / n_evaluable_all:.1f} %)")

    concordance_complete = concordance_df[concordance_df["patient_id"].isin(common)].copy()
    concordance_complete.to_csv(
        OUT_DIR / "cross_layer_aliquot_concordance_complete_cohort.csv", index=False
    )
    n_complete = len(concordance_complete)
    n_four_layer_evaluable = int(concordance_complete["four_layer_evaluable"].sum())
    n_four_layer_concordant = int(concordance_complete["four_layer_concordant"].sum())
    if n_four_layer_evaluable != n_complete:
        log("ATTENTION : certains patients de la cohorte complète ne sont pas "
            "évaluables sur les quatre couches dans l'audit des aliquotes.")
    concordance_rate_complete = (
        100.0 * n_four_layer_concordant / n_four_layer_evaluable
        if n_four_layer_evaluable > 0 else float("nan")
    )
    log(f"Concordance des aliquotes dans la cohorte finale : "
        f"{n_four_layer_concordant}/{n_four_layer_evaluable} "
        f"({concordance_rate_complete:.1f} %)")
else:
    missing_selection_layers = sorted(set(LAYERS) - set(CROSS_LAYER_SELECTIONS.keys()))
    log(f"Audit de concordance inter-couches impossible. "
        f"Sélections manquantes : {missing_selection_layers}")
    concordance_df = pd.DataFrame()
    concordance_complete = pd.DataFrame()
    n_four_layer_evaluable = 0
    n_four_layer_concordant = 0
    concordance_rate_complete = float("nan")


# =============================================================================
# 6. SPLITS DE VALIDATION CROISÉE PARTAGÉS
# =============================================================================

NF = int(CONFIG["cv_folds"])
NR = int(CONFIG["cv_repeats"])
splitter = RepeatedStratifiedKFold(n_splits=NF, n_repeats=NR, random_state=SEED)
SPLITS: List[Tuple[np.ndarray, np.ndarray]] = list(splitter.split(np.zeros((N, 1)), y))
REPEAT_OF = [i // NF for i in range(len(SPLITS))]
FOLD_OF = [i % NF for i in range(len(SPLITS))]
log(f"{len(SPLITS)} splits = {NR} répétitions x {NF} folds")

with open(OUT_DIR / "cv_splits.json", "w", encoding="utf-8") as f:
    json.dump([
        {"split": i, "repeat": REPEAT_OF[i], "fold": FOLD_OF[i],
         "train_indices": tr.tolist(), "test_indices": te.tolist()}
        for i, (tr, te) in enumerate(SPLITS)
    ], f, indent=2)


# =============================================================================
# 7. PRÉTRAITEMENT INTRA-FOLD
# =============================================================================

def score_features(df: pd.DataFrame, stat: str) -> pd.Series:
    if stat == "mad":
        medians = df.median(axis=0)
        return (df - medians).abs().median(axis=0)
    if stat == "iqr":
        quantiles = df.quantile([0.25, 0.75], axis=0)
        return quantiles.loc[0.75] - quantiles.loc[0.25]
    if stat == "var":
        return df.var(axis=0)
    raise ValueError(f"Statistique de dispersion inconnue : {stat}")


def fit_reducer(X_train: np.ndarray, layer: str, seed: int):
    max_allowed = min(X_train.shape[0] - 1, X_train.shape[1] - 1)
    if max_allowed < 1:
        raise ValueError(f"Réduction impossible pour {layer}, shape={X_train.shape}")
    Reducer = TruncatedSVD if layer in BINARY_LAYERS else PCA
    if CONFIG["dim_mode"] == "fixed_k":
        n_components = min(int(CONFIG["dim_k"]), max_allowed)
        reducer = Reducer(n_components=n_components, random_state=seed)
        reducer.fit(X_train)
        return reducer, n_components, float(reducer.explained_variance_ratio_.sum())
    if CONFIG["dim_mode"] == "fixed_variance":
        k_max = min(int(CONFIG["dim_k_max"]), max_allowed)
        probe = Reducer(n_components=k_max, random_state=seed)
        probe.fit(X_train)
        cumulative_variance = np.cumsum(probe.explained_variance_ratio_)
        target = float(CONFIG["dim_variance_target"])
        if cumulative_variance[-1] >= target:
            n_components = int(np.searchsorted(cumulative_variance, target) + 1)
        else:
            n_components = k_max
            miss = VAR_MISS.setdefault(layer, {
                "occurrences": 0, "variance_min": 1.0, "variance_max": 0.0, "components_cap": k_max,
            })
            miss["occurrences"] += 1
            miss["variance_min"] = min(miss["variance_min"], float(cumulative_variance[-1]))
            miss["variance_max"] = max(miss["variance_max"], float(cumulative_variance[-1]))
        reducer = Reducer(n_components=n_components, random_state=seed)
        reducer.fit(X_train)
        return reducer, n_components, float(reducer.explained_variance_ratio_.sum())
    raise ValueError(f"Mode de réduction inconnu : {CONFIG['dim_mode']}")


def preprocess_layer(layer: str, train_index: np.ndarray, test_index: np.ndarray, seed: int) -> Dict[str, Any]:
    df = RAW[layer]
    train_df = df.iloc[train_index].copy()
    test_df = df.iloc[test_index].copy()
    metadata: Dict[str, Any] = {
        "layer": layer, "n_features_initial": int(train_df.shape[1]),
        "n_train": int(len(train_index)), "n_test": int(len(test_index)),
    }

    if layer == "rppa":
        missing_rate = train_df.isna().mean(axis=0)
        keep_columns = missing_rate[missing_rate <= float(CONFIG["rppa_max_missing"])].index
        train_df = train_df.loc[:, keep_columns]
        test_df = test_df.loc[:, keep_columns]
        metadata["n_features_after_missingness"] = int(train_df.shape[1])
        metadata["missing_train_before"] = int(train_df.isna().sum().sum())
        metadata["missing_test_before"] = int(test_df.isna().sum().sum())
        if CONFIG["rppa_imputer"] == "knn":
            imputer = KNNImputer(n_neighbors=int(CONFIG["rppa_knn_k"]))
        elif CONFIG["rppa_imputer"] == "median":
            imputer = SimpleImputer(strategy="median")
        else:
            raise ValueError(f"Imputeur RPPA inconnu : {CONFIG['rppa_imputer']}")
        n_columns_before = train_df.shape[1]
        train_array = imputer.fit_transform(train_df)
        test_array = imputer.transform(test_df)
        if train_array.shape[1] != n_columns_before:
            raise RuntimeError(
                f"[rppa] l'imputation a modifié le nombre de variables : "
                f"{n_columns_before} -> {train_array.shape[1]}"
            )
        train_df = pd.DataFrame(train_array, index=train_df.index, columns=train_df.columns)
        test_df = pd.DataFrame(test_array, index=test_df.index, columns=test_df.columns)
    elif train_df.isna().any().any() or test_df.isna().any().any():
        imputer = SimpleImputer(strategy="median")
        train_df = pd.DataFrame(imputer.fit_transform(train_df), index=train_df.index, columns=train_df.columns)
        test_df = pd.DataFrame(imputer.transform(test_df), index=test_df.index, columns=test_df.columns)

    if layer == "mutations":
        mutation_frequency = train_df.mean(axis=0)
        keep_columns = mutation_frequency[
            (mutation_frequency >= float(CONFIG["mut_freq_lo"]))
            & (mutation_frequency <= float(CONFIG["mut_freq_hi"]))
        ].index
        train_df = train_df.loc[:, keep_columns]
        test_df = test_df.loc[:, keep_columns]
        metadata["n_features_after_frequency"] = int(train_df.shape[1])

    variances = train_df.var(axis=0)
    keep_columns = variances[variances > float(CONFIG["min_variance"])].index
    train_df = train_df.loc[:, keep_columns]
    test_df = test_df.loc[:, keep_columns]
    metadata["n_features_after_variance"] = int(train_df.shape[1])

    if layer in {"mrna", "cnv"} and train_df.shape[1] > int(CONFIG["filter_top_k"]):
        feature_scores = score_features(train_df, str(CONFIG["filter_stat"]))
        selected_columns = feature_scores.nlargest(int(CONFIG["filter_top_k"])).index
        train_df = train_df.loc[:, selected_columns]
        test_df = test_df.loc[:, selected_columns]

    if train_df.shape[1] < 2:
        raise ValueError(f"{layer}: moins de deux variables après filtrage.")

    metadata["n_features_selected"] = int(train_df.shape[1])
    metadata["selected_features"] = train_df.columns.astype(str).tolist()
    X_train = train_df.to_numpy(dtype=np.float64, copy=True)
    X_test = test_df.to_numpy(dtype=np.float64, copy=True)
    assert_finite_matrix(X_train, f"{layer}/train avant standardisation-réduction")
    assert_finite_matrix(X_test, f"{layer}/test avant standardisation-réduction")

    if layer not in BINARY_LAYERS:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        metadata["standardized"] = True
    else:
        metadata["standardized"] = False

    assert_finite_matrix(X_train, f"{layer}/train après standardisation")
    assert_finite_matrix(X_test, f"{layer}/test après standardisation")

    reducer, n_components, retained_variance = fit_reducer(X_train, layer=layer, seed=seed)
    Z_train = reducer.transform(X_train).astype(np.float32)
    Z_test = reducer.transform(X_test).astype(np.float32)
    assert_finite_matrix(Z_train, f"{layer}/train après réduction")
    assert_finite_matrix(Z_test, f"{layer}/test après réduction")

    metadata["n_components"] = int(n_components)
    metadata["retained_variance"] = float(retained_variance)
    metadata["dim_mode"] = CONFIG["dim_mode"]
    metadata["filter_stat"] = CONFIG["filter_stat"]

    return {"X_train": Z_train, "X_test": Z_test, "metadata": metadata}


# =============================================================================
# 8. CONSTRUCTION DU CACHE
# =============================================================================

cache = ckpt_load("preprocessing_cache")
if cache is None:
    cache = {"representations": {}, "metadata": {}}
REPRESENTATIONS = cache["representations"]
PREPROCESS_METADATA = cache["metadata"]

for split_index, (train_index, test_index) in enumerate(SPLITS):
    split_complete = all((split_index, layer) in REPRESENTATIONS for layer in LAYERS)
    if split_complete:
        continue
    log(f"Split {split_index + 1}/{len(SPLITS)}")
    for layer in LAYERS:
        result = preprocess_layer(
            layer=layer, train_index=train_index, test_index=test_index,
            seed=SEED + REPEAT_OF[split_index],
        )
        REPRESENTATIONS[(split_index, layer)] = (result["X_train"], result["X_test"])
        PREPROCESS_METADATA[(split_index, layer)] = result["metadata"]
    if (split_index + 1) % int(CONFIG["cache_save_every"]) == 0 or split_index == len(SPLITS) - 1:
        ckpt_save("preprocessing_cache", {
            "representations": REPRESENTATIONS, "metadata": PREPROCESS_METADATA,
        })


# =============================================================================
# 9. EXPORTS ET RAPPORT FINAL
# =============================================================================

summary_rows = []
for layer in LAYERS:
    rows = [PREPROCESS_METADATA[(i, layer)] for i in range(len(SPLITS))]
    n_features = [r["n_features_selected"] for r in rows]
    n_components = [r["n_components"] for r in rows]
    retained_variance = [r["retained_variance"] for r in rows]
    summary_row = {
        "layer": layer,
        "features_median": int(np.median(n_features)),
        "features_min": int(np.min(n_features)),
        "features_max": int(np.max(n_features)),
        "components_median": int(np.median(n_components)),
        "components_min": int(np.min(n_components)),
        "components_max": int(np.max(n_components)),
        "retained_variance_mean": float(np.mean(retained_variance)),
        "retained_variance_sd": float(np.std(retained_variance, ddof=1)),
    }
    summary_rows.append(summary_row)
    log(f"{SHORT[layer]:<12} features={summary_row['features_median']} "
        f"components={summary_row['components_median']} "
        f"variance={100 * summary_row['retained_variance_mean']:.1f}%")

pd.DataFrame(summary_rows).to_csv(OUT_DIR / "representation_summary.csv", index=False)

flat_metadata = []
for (split_index, layer), metadata in PREPROCESS_METADATA.items():
    flat_metadata.append({
        "split": int(split_index), "repeat": int(REPEAT_OF[split_index]),
        "fold": int(FOLD_OF[split_index]),
        **{k: v for k, v in metadata.items() if k != "selected_features"},
    })
pd.DataFrame(flat_metadata).to_csv(OUT_DIR / "preprocessing_metadata.csv", index=False)

selected_features_payload = {
    f"split_{i}__{layer}": PREPROCESS_METADATA[(i, layer)]["selected_features"]
    for i in range(len(SPLITS)) for layer in LAYERS
}
with gzip.open(OUT_DIR / "selected_features_by_fold.json.gz", "wt", encoding="utf-8") as f:
    json.dump(selected_features_payload, f)

report = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "config": CONFIG_CANONICAL,
    "environment": ENV,
    "inputs": INPUTS,
    "mrna": {
        "source": "UCSC Xena GDC STAR-TPM matrix",
        "declared_scale": CONFIG["mrna_scale"],
        "log_transform_applied_in_this_script": False,
        "scale_audit": MRNA_SCALE_AUDIT,
        "pam50_genes_present": 50,
    },
    "cohort": {
        "n_patients": int(N), "classes": class_counts,
        "layers": {layer: {"n_patients": int(RAW[layer].shape[0]),
                           "n_features": int(RAW[layer].shape[1])} for layer in LAYERS},
    },
    "cross_layer_aliquot_concordance_complete_cohort": {
        "n_patients": int(len(common)),
        "n_four_layer_evaluable": int(n_four_layer_evaluable),
        "n_four_layer_concordant": int(n_four_layer_concordant),
        "concordance_rate_percent": (
            None if not np.isfinite(concordance_rate_complete)
            else float(concordance_rate_complete)
        ),
    },
    "cross_validation": {"folds": NF, "repeats": NR, "n_splits": len(SPLITS)},
    "representation_summary": summary_rows,
    "variance_target_misses": VAR_MISS,
    "design_notes": [
        "Only TCGA primary solid tumor samples (sample type code 01) are retained for all four omics layers.",
        "One primary barcode per patient and per layer is selected deterministically using the alphabetically first barcode.",
        "Cross-layer aliquot concordance is audited rather than enforced.",
        "The mRNA source is a UCSC Xena STAR-TPM matrix already expressed as log2(TPM + 1); no additional logarithmic transformation is applied.",
        "Presence of all 50 PAM50 genes is asserted after mRNA reconstruction and when reloading the mRNA cache.",
        "All data-dependent preprocessing is fitted exclusively on training rows.",
        "The same layer representation is reused unchanged across all combinations within each split.",
        (
            f"The primary analysis uses fixed-k={CONFIG['dim_k']} components per layer."
            if CONFIG["dim_mode"] == "fixed_k"
            else f"The primary analysis uses fixed-variance "
                 f"(target={CONFIG['dim_variance_target']}, cap={CONFIG['dim_k_max']}) components per layer."
        ),
        "mRNA and CNV filtering is unsupervised and based on training-fold MAD.",
        "PAM50 labels are not used for feature selection or dimensionality reduction.",
        "Checkpoints are invalidated if either the configuration or any input file SHA-256 changes.",
        "Defensive checks reject exact barcode duplicates and non-finite matrices.",
        f"RUN_ID='{RUN_ID}' -- écrit dans un dossier séparé du run k=50, sans écrasement possible.",
    ],
}

with open(OUT_DIR / "preprocessing_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
with open(OUT_DIR / "preprocessing_run.log", "w", encoding="utf-8") as f:
    f.write("\n".join(LOG))

log("")
log("Prétraitement terminé avec succès.")
log(f"Résultats : {OUT_DIR}")

#2.Evaluation du PCA 100

Le script part des données déjà réduites (les 50 splits × 4 couches en 50 ou 100 composantes, calculés par le prétraitement) et teste toutes les combinaisons possibles de couches, avec trois algorithmes différents, pour répondre à la question : quelles couches omiques, seules ou combinées, prédisent le mieux le sous-type PAM50, et est-ce que la réponse dépend de l'algorithme utilisé ?
En résumé, ce que le programme répond concrètement
###1.Quelle couche domine ? => Shapley (mRNA, de loin, dans les 3 classifieurs)
###2.Quelle combinaison est minimale mais suffisante ? => mRNA seul (accord unanime)
###3.Combiner des couches aide-t-il vraiment, statistiquement ? => Oui mais seulement pour XGBoost, sur 3 panels précis
###4.Les classifieurs sont-ils d'accord entre eux sur le classement des 15 panels ? => Partiellement (RF-XGBoost très alignés, SVM un peu à part)

In [ ]:
"""
===============================================================================
 — ÉVALUATION DES 15 PANELS OMIQUES (k100)
===============================================================================

"""

import os
import json
import math
import time
import warnings
from pathlib import Path
from itertools import combinations
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =============================================================================
# 0. VÉRIFICATION DE L'ÉTAT FOURNI PAR LE PRÉTRAITEMENT k100
# =============================================================================

REQUIRED_GLOBALS = [
    "CONFIG", "SEED", "LAYERS", "SHORT", "SPLITS", "REPEAT_OF", "FOLD_OF",
    "REPRESENTATIONS", "y", "N", "OUT_DIR", "RUN_ID",
]
missing_globals = [name for name in REQUIRED_GLOBALS if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Ce module doit être exécuté après le script de prétraitement k100, "
        f"dans la même session. Variables manquantes : {missing_globals}"
    )

if RUN_ID != "k100":
    raise RuntimeError(
        f"RUN_ID courant = '{RUN_ID}', attendu 'k100'. "
        "Relancez le prétraitement k100 avant ce module pour éviter "
        "d'écrire les résultats d'évaluation dans le mauvais dossier."
    )

if int(CONFIG["dim_k"]) != 100:
    raise RuntimeError(
        f"CONFIG['dim_k'] = {CONFIG['dim_k']}, attendu 100. "
        "Le cache en mémoire ne correspond pas au régime k100."
    )

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR = OUT_DIR / "model_evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUN_ID confirmé : {RUN_ID}")
print(f"dim_k confirmé : {CONFIG['dim_k']}")
print(f"Résultats d'évaluation écrits dans : {EVAL_DIR}")

EVAL_CONFIG: Dict[str, Any] = {
    "primary_metric": "balanced_accuracy",
    "bootstrap_replicates_repeat_level": 10000,
    "fdr_alpha": 0.05,
    "minimal_panel_delta": 0.03,
    "random_forest": {
        "n_estimators": 500,
        "max_features": "sqrt",
        "min_samples_leaf": 2,
        "n_jobs": -1,
    },
    "xgboost": {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0,
        "min_child_weight": 1.0,
        "n_jobs": -1,
    },
    "svm": {
        "C": 1.0,
        "kernel": "rbf",
        "gamma": "scale",
        "class_weight": "balanced",
        "probability": False,
    },
}


def eval_log(message: str) -> None:
    print(message)


# =============================================================================
# 1. PANELS : LATTICE COMPLET DES 4 COUCHES
# =============================================================================

def canonical_panel(panel: Tuple[str, ...]) -> Tuple[str, ...]:
    order = {layer: i for i, layer in enumerate(LAYERS)}
    return tuple(sorted(panel, key=lambda x: order[x]))


PANELS: List[Tuple[str, ...]] = []
for size in range(1, len(LAYERS) + 1):
    PANELS.extend(canonical_panel(tuple(p)) for p in combinations(LAYERS, size))

FULL_PANEL = tuple(LAYERS)
if len(PANELS) != 15:
    raise RuntimeError(f"15 panels attendus, obtenu : {len(PANELS)}")


def panel_key(panel: Tuple[str, ...]) -> str:
    return "+".join(panel)


def panel_label(panel: Tuple[str, ...]) -> str:
    return "+".join(SHORT[layer] for layer in panel)


# =============================================================================
# 2. CONSTRUCTION DES MATRICES PAR PANEL
# =============================================================================

def get_panel_matrices(split_index: int, panel: Tuple[str, ...]) -> Tuple[np.ndarray, np.ndarray]:
    train_blocks, test_blocks = [], []
    for layer in panel:
        key = (split_index, layer)
        if key not in REPRESENTATIONS:
            raise KeyError(f"Représentation absente : {key}")
        Xtr, Xte = REPRESENTATIONS[key]
        train_blocks.append(Xtr)
        test_blocks.append(Xte)
    X_train = np.concatenate(train_blocks, axis=1)
    X_test = np.concatenate(test_blocks, axis=1)
    if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
        raise RuntimeError(f"Valeurs non finies : panel {panel_key(panel)}, split {split_index}.")
    return X_train, X_test


# =============================================================================
# 3. CLASSIFIEURS PRÉSPÉCIFIÉS
# =============================================================================

def make_classifier(classifier_name: str, seed: int, n_classes: int):
    if classifier_name == "RF":
        return RandomForestClassifier(random_state=seed, **EVAL_CONFIG["random_forest"])
    if classifier_name == "XGB":
        return XGBClassifier(
            objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss",
            random_state=seed, verbosity=0, **EVAL_CONFIG["xgboost"],
        )
    if classifier_name == "SVM":
        return SVC(random_state=seed, **EVAL_CONFIG["svm"])
    if classifier_name == "DUMMY":
        return DummyClassifier(strategy="prior", random_state=seed)
    raise ValueError(f"Classifieur inconnu : {classifier_name}")


CLASSIFIERS = ["RF", "XGB", "SVM"]


# =============================================================================
# 4. ÉVALUATION DES 15 PANELS + BASELINE -- REPRISE FINE
# =============================================================================

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }


fold_results_path = EVAL_DIR / "panel_fold_results.csv"

if fold_results_path.exists():
    fold_results_df = pd.read_csv(fold_results_path)
    eval_log(f"Rechargement partiel : {fold_results_path.name} "
             f"({len(fold_results_df)} lignes déjà présentes)")
else:
    fold_results_df = pd.DataFrame()

done_split_classifier = set()
if not fold_results_df.empty:
    counts = fold_results_df.groupby(["split", "classifier"]).size()
    expected_rows_per_split_classifier = len(PANELS) + 1
    for (split_index, classifier_name), n_rows in counts.items():
        if n_rows >= expected_rows_per_split_classifier:
            done_split_classifier.add((int(split_index), classifier_name))

rows: List[Dict[str, Any]] = fold_results_df.to_dict("records") if not fold_results_df.empty else []
n_classes = int(len(np.unique(y)))
start_time = time.time()
n_total = len(SPLITS) * len(CLASSIFIERS)
n_done_at_start = len(done_split_classifier)
eval_log(f"Reprise : {n_done_at_start}/{n_total} (split, classifier) déjà complets")

for split_index, (train_index, test_index) in enumerate(SPLITS):
    repeat_index = int(REPEAT_OF[split_index])
    fold_index = int(FOLD_OF[split_index])
    classifier_seed = int(SEED + repeat_index)
    y_train = y[train_index]
    y_test = y[test_index]

    any_new_this_split = False

    for classifier_name in CLASSIFIERS:
        if (split_index, classifier_name) in done_split_classifier:
            continue
        any_new_this_split = True

        dummy = make_classifier("DUMMY", seed=classifier_seed, n_classes=n_classes)
        dummy.fit(np.zeros((len(train_index), 1), dtype=np.float32), y_train)
        dummy_pred = dummy.predict(np.zeros((len(test_index), 1), dtype=np.float32))
        dummy_metrics = compute_metrics(y_test, dummy_pred)
        rows.append({
            "classifier": classifier_name, "split": split_index, "repeat": repeat_index,
            "fold": fold_index, "panel": "EMPTY", "panel_label": "Baseline", "panel_size": 0,
            "balanced_accuracy": dummy_metrics["balanced_accuracy"],
            "macro_f1": dummy_metrics["macro_f1"], "accuracy": dummy_metrics["accuracy"],
            "fit_seconds": 0.0,
        })

        sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

        for panel in PANELS:
            X_train, X_test = get_panel_matrices(split_index, panel)
            model = make_classifier(classifier_name, seed=classifier_seed, n_classes=n_classes)

            fit_start = time.time()
            if classifier_name in {"RF", "XGB"}:
                model.fit(X_train, y_train, sample_weight=sample_weight)
            else:
                model.fit(X_train, y_train)
            fit_seconds = time.time() - fit_start

            predictions = model.predict(X_test)
            metrics = compute_metrics(y_test, predictions)
            rows.append({
                "classifier": classifier_name, "split": split_index, "repeat": repeat_index,
                "fold": fold_index, "panel": panel_key(panel), "panel_label": panel_label(panel),
                "panel_size": len(panel), "balanced_accuracy": metrics["balanced_accuracy"],
                "macro_f1": metrics["macro_f1"], "accuracy": metrics["accuracy"],
                "fit_seconds": float(fit_seconds),
            })

        done_split_classifier.add((split_index, classifier_name))

    if any_new_this_split:
        tmp_path = fold_results_path.with_suffix(".csv.tmp")
        pd.DataFrame(rows).to_csv(tmp_path, index=False)
        os.replace(tmp_path, fold_results_path)
        eval_log(f"  progression : {len(done_split_classifier)}/{n_total} "
                 f"(split, classifier) -- ({time.time() - start_time:.0f}s cette session)")

fold_results = pd.DataFrame(rows)
eval_log(f"Résultats fold-level : {len(fold_results)} lignes "
         f"({len(done_split_classifier)}/{n_total} (split, classifier) couverts)")

if len(done_split_classifier) != n_total:
    raise RuntimeError(
        f"Évaluation incomplète : {len(done_split_classifier)}/{n_total} "
        "(split, classifier). Relancez ce script pour terminer avant de continuer."
    )


# =============================================================================
# 5. AGRÉGATION AU NIVEAU DES RÉPÉTITIONS
# =============================================================================

METRICS = ["balanced_accuracy", "macro_f1", "accuracy"]

repeat_results = (
    fold_results.groupby(["classifier", "repeat", "panel", "panel_label", "panel_size"], as_index=False)[METRICS]
    .mean()
)
repeat_results.to_csv(EVAL_DIR / "panel_repeat_results.csv", index=False)

panel_summary = (
    repeat_results.groupby(["classifier", "panel", "panel_label", "panel_size"], as_index=False)[METRICS]
    .agg(["mean", "std"])
)
panel_summary.columns = ["_".join(str(x) for x in c if str(x) != "") for c in panel_summary.columns.to_flat_index()]
panel_summary.to_csv(EVAL_DIR / "panel_summary.csv", index=False)


# =============================================================================
# 6. OUTILS STATISTIQUES AU NIVEAU RÉPÉTITION
# =============================================================================

def bootstrap_mean_ci(values: np.ndarray, n_boot: int, seed: int, alpha: float = 0.05) -> Tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    draws = rng.choice(values, size=(n_boot, values.size), replace=True)
    means = draws.mean(axis=1)
    return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))


def exact_sign_flip_pvalue_greater(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan")
    observed = float(values.mean())
    n = values.size
    permuted_means = []
    for mask in range(1 << n):
        signs = np.ones(n, dtype=float)
        for i in range(n):
            if (mask >> i) & 1:
                signs[i] = -1.0
        permuted_means.append(float(np.mean(values * signs)))
    permuted_means = np.asarray(permuted_means)
    return float((np.sum(permuted_means >= observed) + 1) / (len(permuted_means) + 1))


# =============================================================================
# 7. GAIN DIRECT AU-DELÀ DE LA MEILLEURE COUCHE CONSTITUANTE
# =============================================================================

primary_metric = str(EVAL_CONFIG["primary_metric"])

repeat_lookup = {
    (row.classifier, int(row.repeat), row.panel): float(getattr(row, primary_metric))
    for row in repeat_results.itertuples(index=False)
}

gain_rows: List[Dict[str, Any]] = []
for classifier_name in CLASSIFIERS:
    for repeat_index in sorted(repeat_results["repeat"].unique()):
        for panel in PANELS:
            if len(panel) < 2:
                continue
            panel_name = panel_key(panel)
            panel_score = repeat_lookup[(classifier_name, int(repeat_index), panel_name)]
            constituent_scores = {
                layer: repeat_lookup[(classifier_name, int(repeat_index), panel_key((layer,)))]
                for layer in panel
            }
            best_single_layer = max(constituent_scores, key=constituent_scores.get)
            best_single_score = constituent_scores[best_single_layer]
            gain = panel_score - best_single_score
            gain_rows.append({
                "classifier": classifier_name, "repeat": int(repeat_index), "panel": panel_name,
                "panel_label": panel_label(panel), "panel_size": len(panel),
                "panel_score": float(panel_score), "best_constituent_single": best_single_layer,
                "best_constituent_single_score": float(best_single_score),
                "integration_gain": float(gain), "gain_positive": bool(gain > 0),
            })

integration_gains = pd.DataFrame(gain_rows)
integration_gains.to_csv(EVAL_DIR / "integration_gain_repeat_level.csv", index=False)

gain_summary_rows = []
for (classifier_name, panel_name, panel_display, panel_size), group in integration_gains.groupby(
    ["classifier", "panel", "panel_label", "panel_size"]
):
    values = group["integration_gain"].to_numpy(dtype=float)
    ci_low, ci_high = bootstrap_mean_ci(
        values, n_boot=int(EVAL_CONFIG["bootstrap_replicates_repeat_level"]),
        seed=SEED + sum(ord(c) for c in classifier_name + panel_name),
    )
    p_value = exact_sign_flip_pvalue_greater(values)
    gain_summary_rows.append({
        "classifier": classifier_name, "panel": panel_name, "panel_label": panel_display,
        "panel_size": int(panel_size), "mean_gain": float(np.mean(values)),
        "sd_gain": float(np.std(values, ddof=1)), "ci95_low": ci_low, "ci95_high": ci_high,
        "positive_gain_frequency": float(np.mean(values > 0)), "n_repeats": int(len(values)),
        "p_signflip_greater": p_value,
    })

gain_summary = pd.DataFrame(gain_summary_rows)
gain_summary["q_bh"] = np.nan
gain_summary["significant_positive_gain"] = False
for classifier_name, indices in gain_summary.groupby("classifier").groups.items():
    p_values = gain_summary.loc[indices, "p_signflip_greater"].to_numpy(dtype=float)
    reject, q_values, _, _ = multipletests(p_values, alpha=float(EVAL_CONFIG["fdr_alpha"]), method="fdr_bh")
    gain_summary.loc[indices, "q_bh"] = q_values
    gain_summary.loc[indices, "significant_positive_gain"] = (
        reject & (gain_summary.loc[indices, "mean_gain"].to_numpy(dtype=float) > 0)
    )
gain_summary.to_csv(EVAL_DIR / "integration_gain_summary.csv", index=False)


# =============================================================================
# 8. SHAPLEY EXACT DE PERFORMANCE
# =============================================================================

def all_subsets_without(items: Tuple[str, ...], excluded: str) -> List[Tuple[str, ...]]:
    remaining = tuple(item for item in items if item != excluded)
    subsets = [tuple()]
    for size in range(1, len(remaining) + 1):
        subsets.extend(canonical_panel(tuple(s)) for s in combinations(remaining, size))
    return subsets


def exact_performance_shapley(value_function: Dict[Tuple[str, ...], float]) -> Dict[str, float]:
    n = len(LAYERS)
    shapley = {layer: 0.0 for layer in LAYERS}
    for layer in LAYERS:
        for subset in all_subsets_without(tuple(LAYERS), layer):
            subset = canonical_panel(subset)
            with_layer = canonical_panel(subset + (layer,))
            if subset not in value_function or with_layer not in value_function:
                raise KeyError(f"Valeur absente pour {subset} ou {with_layer}")
            s = len(subset)
            weight = math.factorial(s) * math.factorial(n - s - 1) / math.factorial(n)
            shapley[layer] += weight * (value_function[with_layer] - value_function[subset])
    return shapley


shapley_repeat_rows = []
for classifier_name in CLASSIFIERS:
    for repeat_index in sorted(repeat_results["repeat"].unique()):
        value_function: Dict[Tuple[str, ...], float] = {}
        value_function[tuple()] = repeat_lookup[(classifier_name, int(repeat_index), "EMPTY")]
        for panel in PANELS:
            value_function[panel] = repeat_lookup[(classifier_name, int(repeat_index), panel_key(panel))]
        shapley_values = exact_performance_shapley(value_function)
        total_shapley = sum(shapley_values.values())
        total_gain = value_function[FULL_PANEL] - value_function[tuple()]
        if not np.isclose(total_shapley, total_gain, atol=1e-8):
            raise RuntimeError(
                f"Échec propriété d'efficacité Shapley : {classifier_name}, "
                f"repeat={repeat_index}, {total_shapley} != {total_gain}"
            )
        for layer, value in shapley_values.items():
            shapley_repeat_rows.append({
                "classifier": classifier_name, "repeat": int(repeat_index), "layer": layer,
                "layer_label": SHORT[layer], "performance_shapley": float(value),
                "full_minus_baseline": float(total_gain),
            })

shapley_repeat = pd.DataFrame(shapley_repeat_rows)
shapley_repeat.to_csv(EVAL_DIR / "performance_shapley_repeat_level.csv", index=False)

shapley_summary_rows = []
for (classifier_name, layer, layer_label), group in shapley_repeat.groupby(["classifier", "layer", "layer_label"]):
    values = group["performance_shapley"].to_numpy(dtype=float)
    ci_low, ci_high = bootstrap_mean_ci(
        values, n_boot=int(EVAL_CONFIG["bootstrap_replicates_repeat_level"]),
        seed=SEED + sum(ord(c) for c in classifier_name + layer),
    )
    shapley_summary_rows.append({
        "classifier": classifier_name, "layer": layer, "layer_label": layer_label,
        "mean_performance_shapley": float(np.mean(values)), "sd_performance_shapley": float(np.std(values, ddof=1)),
        "ci95_low": ci_low, "ci95_high": ci_high, "positive_frequency": float(np.mean(values > 0)),
    })
shapley_summary = pd.DataFrame(shapley_summary_rows)
shapley_summary.to_csv(EVAL_DIR / "performance_shapley_summary.csv", index=False)


# =============================================================================
# 9. CLASSEMENT ET ROBUSTESSE INTER-CLASSIFIEUR
# =============================================================================

panel_mean_scores = (
    repeat_results[repeat_results["panel"] != "EMPTY"]
    .groupby(["classifier", "panel", "panel_label", "panel_size"], as_index=False)[primary_metric]
    .mean().rename(columns={primary_metric: "mean_score"})
)
panel_mean_scores["rank_within_classifier"] = (
    panel_mean_scores.groupby("classifier")["mean_score"].rank(method="average", ascending=False)
)
panel_mean_scores.to_csv(EVAL_DIR / "panel_classifier_ranking.csv", index=False)

rank_pivot = panel_mean_scores.pivot(index="panel", columns="classifier", values="rank_within_classifier")
rank_correlation_rows = []
for classifier_a, classifier_b in combinations(CLASSIFIERS, 2):
    rho, p_value = spearmanr(rank_pivot[classifier_a], rank_pivot[classifier_b])
    rank_correlation_rows.append({
        "classifier_a": classifier_a, "classifier_b": classifier_b,
        "spearman_rho": float(rho), "p_value": float(p_value),
    })
rank_correlations = pd.DataFrame(rank_correlation_rows)
rank_correlations.to_csv(EVAL_DIR / "classifier_rank_correlations.csv", index=False)

robust_panel_summary = (
    panel_mean_scores.groupby(["panel", "panel_label", "panel_size"], as_index=False)
    .agg(
        mean_score_across_classifiers=("mean_score", "mean"),
        sd_score_across_classifiers=("mean_score", "std"),
        mean_rank_across_classifiers=("rank_within_classifier", "mean"),
        worst_rank_across_classifiers=("rank_within_classifier", "max"),
        top3_frequency=("rank_within_classifier", lambda x: float(np.mean(x <= 3))),
    )
    .sort_values(["mean_rank_across_classifiers", "mean_score_across_classifiers"], ascending=[True, False])
)
robust_panel_summary.to_csv(EVAL_DIR / "panel_robustness_across_classifiers.csv", index=False)


# =============================================================================
# 10. PANEL MINIMAL SUFFISANT -- GLOBAL ET PAR CLASSIFIEUR
# =============================================================================

best_overall_score = float(robust_panel_summary["mean_score_across_classifiers"].max())
minimal_threshold = best_overall_score - float(EVAL_CONFIG["minimal_panel_delta"])
minimal_candidates = robust_panel_summary[
    robust_panel_summary["mean_score_across_classifiers"] >= minimal_threshold
].copy()
minimal_candidates = minimal_candidates.sort_values(
    ["panel_size", "mean_rank_across_classifiers", "mean_score_across_classifiers"],
    ascending=[True, True, False],
)
minimal_sufficient_panel_row = minimal_candidates.iloc[0] if not minimal_candidates.empty else None
minimal_candidates.to_csv(EVAL_DIR / "minimal_sufficient_panel_candidates.csv", index=False)

minimal_panel_per_classifier_rows = []
for classifier_name in CLASSIFIERS:
    subset = panel_mean_scores[panel_mean_scores["classifier"] == classifier_name]
    best_score_clf = float(subset["mean_score"].max())
    threshold_clf = best_score_clf - float(EVAL_CONFIG["minimal_panel_delta"])
    candidates_clf = subset[subset["mean_score"] >= threshold_clf].copy()
    candidates_clf["panel_size"] = candidates_clf["panel"].apply(lambda p: len(p.split("+")))
    candidates_clf = candidates_clf.sort_values(["panel_size", "mean_score"], ascending=[True, False])
    if not candidates_clf.empty:
        best_row = candidates_clf.iloc[0]
        minimal_panel_per_classifier_rows.append({
            "classifier": classifier_name, "panel": best_row["panel"],
            "panel_label": best_row["panel_label"], "panel_size": int(best_row["panel_size"]),
            "mean_score": float(best_row["mean_score"]),
        })

minimal_panel_per_classifier = pd.DataFrame(minimal_panel_per_classifier_rows)
minimal_panel_per_classifier.to_csv(EVAL_DIR / "minimal_sufficient_panel_per_classifier.csv", index=False)

panels_agree_across_classifiers = (
    minimal_panel_per_classifier["panel"].nunique() == 1
    if not minimal_panel_per_classifier.empty else None
)
eval_log(f"Panels minimaux par classifieur : "
         f"{minimal_panel_per_classifier[['classifier','panel']].to_dict('records') if not minimal_panel_per_classifier.empty else 'N/A'}")
eval_log(f"Accord inter-classifieur sur le panel minimal : {panels_agree_across_classifiers}")


# =============================================================================
# 11. SYNTHÈSE DES RÉPONSES PRIMAIRES
# =============================================================================

def best_panel_of_size(size: int) -> Optional[pd.Series]:
    subset = robust_panel_summary[robust_panel_summary["panel_size"] == size]
    if subset.empty:
        return None
    return subset.sort_values(
        ["mean_score_across_classifiers", "mean_rank_across_classifiers"], ascending=[False, True]
    ).iloc[0]


best_single = best_panel_of_size(1)
best_pair = best_panel_of_size(2)
best_triplet = best_panel_of_size(3)
best_overall = robust_panel_summary.iloc[0]

best_gain_row = gain_summary.sort_values(
    ["mean_gain", "positive_gain_frequency"], ascending=[False, False]
).iloc[0]

dominant_layer_row = (
    shapley_summary.groupby(["layer", "layer_label"], as_index=False)["mean_performance_shapley"]
    .mean().sort_values("mean_performance_shapley", ascending=False).iloc[0]
)

significant_gain_panels = gain_summary[gain_summary["significant_positive_gain"]].copy()
does_multiomics_improve = bool(not significant_gain_panels.empty)

primary_answers = {
    "run_id": RUN_ID,
    "dim_k": int(CONFIG["dim_k"]),
    "primary_metric": primary_metric,
    "does_multiomics_improve_prediction": {
        "answer": does_multiomics_improve,
        "criterion": ("At least one multi-omics panel has a positive repeat-level "
                     "integration gain after BH correction in at least one classifier."),
        "n_significant_panel_classifier_results": int(len(significant_gain_panels)),
    },
    "best_single_omic": None if best_single is None else {
        "panel": str(best_single["panel"]), "panel_label": str(best_single["panel_label"]),
        "mean_score_across_classifiers": float(best_single["mean_score_across_classifiers"]),
        "mean_rank_across_classifiers": float(best_single["mean_rank_across_classifiers"]),
    },
    "best_pair": None if best_pair is None else {
        "panel": str(best_pair["panel"]), "panel_label": str(best_pair["panel_label"]),
        "mean_score_across_classifiers": float(best_pair["mean_score_across_classifiers"]),
        "mean_rank_across_classifiers": float(best_pair["mean_rank_across_classifiers"]),
    },
    "best_triplet": None if best_triplet is None else {
        "panel": str(best_triplet["panel"]), "panel_label": str(best_triplet["panel_label"]),
        "mean_score_across_classifiers": float(best_triplet["mean_score_across_classifiers"]),
        "mean_rank_across_classifiers": float(best_triplet["mean_rank_across_classifiers"]),
    },
    "best_overall_panel": {
        "panel": str(best_overall["panel"]), "panel_label": str(best_overall["panel_label"]),
        "panel_size": int(best_overall["panel_size"]),
        "mean_score_across_classifiers": float(best_overall["mean_score_across_classifiers"]),
        "mean_rank_across_classifiers": float(best_overall["mean_rank_across_classifiers"]),
    },
    "minimal_sufficient_panel_global": None if minimal_sufficient_panel_row is None else {
        "panel": str(minimal_sufficient_panel_row["panel"]),
        "panel_label": str(minimal_sufficient_panel_row["panel_label"]),
        "panel_size": int(minimal_sufficient_panel_row["panel_size"]),
        "mean_score_across_classifiers": float(minimal_sufficient_panel_row["mean_score_across_classifiers"]),
        "delta_from_best_threshold": float(EVAL_CONFIG["minimal_panel_delta"]),
    },
    "minimal_sufficient_panel_per_classifier": minimal_panel_per_classifier_rows,
    "minimal_panels_agree_across_classifiers": panels_agree_across_classifiers,
    "most_classifier_robust_panel": {
        "panel": str(best_overall["panel"]), "panel_label": str(best_overall["panel_label"]),
        "mean_rank_across_classifiers": float(best_overall["mean_rank_across_classifiers"]),
        "worst_rank_across_classifiers": float(best_overall["worst_rank_across_classifiers"]),
        "top3_frequency": float(best_overall["top3_frequency"]),
    },
    "largest_incremental_gain": {
        "classifier": str(best_gain_row["classifier"]), "panel": str(best_gain_row["panel"]),
        "panel_label": str(best_gain_row["panel_label"]), "mean_gain": float(best_gain_row["mean_gain"]),
        "ci95_low": float(best_gain_row["ci95_low"]), "ci95_high": float(best_gain_row["ci95_high"]),
        "positive_gain_frequency": float(best_gain_row["positive_gain_frequency"]),
        "q_bh": float(best_gain_row["q_bh"]),
    },
    "most_dominant_layer_by_performance_shapley": {
        "layer": str(dominant_layer_row["layer"]), "layer_label": str(dominant_layer_row["layer_label"]),
        "mean_performance_shapley_across_classifiers": float(dominant_layer_row["mean_performance_shapley"]),
    },
}

with open(EVAL_DIR / "primary_answers.json", "w", encoding="utf-8") as f:
    json.dump(primary_answers, f, indent=2, ensure_ascii=False)
with open(EVAL_DIR / "evaluation_config.json", "w", encoding="utf-8") as f:
    json.dump(EVAL_CONFIG, f, indent=2, ensure_ascii=False)

eval_log("")
eval_log("=" * 80)
eval_log(f"ÉVALUATION DES 15 PANELS TERMINÉE (RUN_ID={RUN_ID}, dim_k={CONFIG['dim_k']})")
eval_log(f"Résultats : {EVAL_DIR}")
eval_log("=" * 80)
eval_log(f"Meilleur panel global : {primary_answers['best_overall_panel']['panel_label']}")
eval_log(f"Panel minimal suffisant (global) : {primary_answers['minimal_sufficient_panel_global']}")
eval_log(f"Panels minimaux par classifieur : {primary_answers['minimal_sufficient_panel_per_classifier']}")
eval_log(f"Accord inter-classifieur : {primary_answers['minimal_panels_agree_across_classifiers']}")
eval_log(f"Couche dominante par Shapley : "
         f"{primary_answers['most_dominant_layer_by_performance_shapley']['layer_label']}")

#Vérification du cache

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data")
except Exception:
    DATA_DIR = Path("./TCGA_BRCA_data")

DATA_DIR = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data")
runs_dir = DATA_DIR / "runs"

print("Dossiers de runs existants :")
for run_folder in sorted(runs_dir.iterdir()):
    if run_folder.is_dir():
        results = run_folder / "results"
        model_eval = results / "model_evaluation"
        print(f"\n{run_folder.name}/")
        print(f"  results/ existe : {results.exists()}")
        print(f"  results/model_evaluation/ existe : {model_eval.exists()}")
        if results.exists():
            print(f"  fichiers dans results/ : {[f.name for f in results.iterdir() if f.is_file()][:5]}")

In [ ]:
import os
os.environ["RUN_ID"] = "2026-07-26"



In [ ]:
"""
Prétraitement pca50 -- RECHARGE le cache déjà construit.
"""

import os
import sys
import json
import gzip
import pickle
import hashlib
import platform
import tarfile
import warnings
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Any, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.model_selection import RepeatedStratifiedKFold

import sklearn
import scipy

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/fati/TCGA_BRCA_data")
except Exception:
    DATA_DIR = Path("./TCGA_BRCA_data")

RUN_ID = os.environ.get("RUN_ID") or datetime.now().strftime("%Y-%m-%d")
RUN_DIR = DATA_DIR / "runs" / RUN_ID
CKPT_DIR = RUN_DIR / "checkpoints"
OUT_DIR = RUN_DIR / "results"

for d in (DATA_DIR, RUN_DIR, CKPT_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

CONFIG: Dict[str, Any] = {
    "cv_folds": 5,
    "cv_repeats": 10,
    "seed": 42,
    "allowed_sample_type_codes": ["01"],
    "duplicate_rule": "first_sorted",
    "filter_stat": "mad",
    "filter_top_k": 5000,
    "min_variance": 1e-6,
    "mut_freq_lo": 0.01,
    "mut_freq_hi": 0.99,
    "rppa_max_missing": 0.20,
    "rppa_imputer": "knn",
    "rppa_knn_k": 5,
    "dim_mode": "fixed_k",
    "dim_k": 50,
    "dim_variance_target": 0.80,
    "dim_k_max": 120,
    "mrna_scale": "log2(TPM + 1)",
    "mrna_apply_log_transform": False,
    "cache_save_every": 5,
}

CONFIG_CANONICAL = deepcopy(CONFIG)
SEED = int(CONFIG["seed"])

LAYERS = ["mutations", "cnv", "mrna", "rppa"]
SHORT = {"mutations": "Mutations", "cnv": "CNV", "mrna": "mRNA", "rppa": "RPPA"}
BINARY_LAYERS = {"mutations"}

NONSYNONYMOUS = {
    "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
    "Frame_Shift_Ins", "In_Frame_Del", "In_Frame_Ins", "Splice_Site",
    "Nonstop_Mutation", "Translation_Start_Site",
}

PAM50_GENES = [
    "ACTR3B", "ANLN", "BAG1", "BCL2", "BIRC5", "BLVRA", "CCNB1",
    "CCNE1", "CDC20", "CDC6", "CDH3", "CENPF", "CEP55", "CXXC5",
    "EGFR", "ERBB2", "ESR1", "EXO1", "FGFR4", "FOXA1", "FOXC1",
    "GPR160", "GRB7", "KIF2C", "KRT14", "KRT17", "KRT5", "MAPT",
    "MDM2", "MELK", "MIA", "MKI67", "MLPH", "MMP11", "MYBL2",
    "MYC", "NAT1", "NDC80", "NUF2", "ORC6", "PGR", "PHGDH",
    "PTTG1", "RRM2", "SFRP1", "SLC39A6", "TMEM45B", "TYMS",
    "UBE2C", "UBE2T",
]

ENV = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "numpy": np.__version__, "pandas": pd.__version__,
    "sklearn": sklearn.__version__, "scipy": scipy.__version__,
}

LOG: List[str] = []
INPUTS: Dict[str, Any] = {}
VAR_MISS: Dict[str, Dict[str, Any]] = {}
CROSS_LAYER_SELECTIONS: Dict[str, Dict[str, str]] = {}


def log(message: str) -> None:
    print(message)
    LOG.append(f"{datetime.now().strftime('%H:%M:%S')}  {message}")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def register_input(name: str, path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Fichier manquant : {path}")
    INPUTS[name] = {
        "path": str(path), "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }


def ckpt_save(name: str, data: Any) -> None:
    target = CKPT_DIR / f"ckpt_{name}.pkl"
    tmp = target.with_suffix(".pkl.tmp")
    payload = {
        "data": data, "run": RUN_ID, "config": CONFIG_CANONICAL,
        "env": ENV, "inputs": deepcopy(INPUTS),
        "written": datetime.now().isoformat(timespec="seconds"),
    }
    with open(tmp, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, target)


def ckpt_load(name: str) -> Optional[Any]:
    path = CKPT_DIR / f"ckpt_{name}.pkl"
    if not path.exists():
        return None
    with open(path, "rb") as f:
        payload = pickle.load(f)
    if payload.get("config") != CONFIG_CANONICAL:
        log(f"[ckpt] {name}: configuration différente, checkpoint ignoré.")
        return None
    if payload.get("inputs") != INPUTS:
        log(f"[ckpt] {name}: fichiers d'entrée différents, checkpoint ignoré.")
        return None
    log(f"[ckpt] {name}: rechargé.")
    return payload["data"]


def assert_finite_matrix(X: np.ndarray, label: str) -> None:
    if X.ndim != 2:
        raise RuntimeError(f"{label}: matrice non bidimensionnelle, shape={X.shape}")
    if X.shape[0] == 0 or X.shape[1] == 0:
        raise RuntimeError(f"{label}: matrice vide, shape={X.shape}")
    if not np.isfinite(X).all():
        n_bad = int((~np.isfinite(X)).sum())
        raise RuntimeError(f"{label}: {n_bad} valeurs NaN ou infinies détectées")


def tcga_patient_id(barcode: str) -> str:
    parts = str(barcode).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(barcode)


def tcga_sample_type_code(barcode: str) -> Optional[str]:
    parts = str(barcode).split("-")
    if len(parts) < 4 or len(parts[3]) < 2:
        return None
    return parts[3][:2]


def tcga_sample_suffix(barcode: Optional[str]) -> Optional[str]:
    if barcode is None:
        return None
    parts = str(barcode).split("-")
    if len(parts) < 4:
        return None
    return parts[3]


def select_primary_tumor_samples(sample_ids: List[str]) -> Dict[str, str]:
    grouped: Dict[str, List[str]] = {}
    allowed = set(CONFIG["allowed_sample_type_codes"])
    for barcode in map(str, sample_ids):
        if tcga_sample_type_code(barcode) not in allowed:
            continue
        grouped.setdefault(tcga_patient_id(barcode), []).append(barcode)
    if CONFIG["duplicate_rule"] != "first_sorted":
        raise ValueError(f"Règle de doublon non implémentée : {CONFIG['duplicate_rule']}")
    return {patient: sorted(barcodes)[0] for patient, barcodes in grouped.items()}


def audit_duplicate_primary_samples(sample_ids: List[str], layer: str) -> pd.DataFrame:
    grouped: Dict[str, List[str]] = {}
    allowed = set(CONFIG["allowed_sample_type_codes"])
    for barcode in map(str, sample_ids):
        if tcga_sample_type_code(barcode) not in allowed:
            continue
        grouped.setdefault(tcga_patient_id(barcode), []).append(barcode)
    records = []
    for patient, barcodes in grouped.items():
        if len(barcodes) > 1:
            ordered = sorted(barcodes)
            records.append({
                "layer": layer, "patient_id": patient,
                "n_primary_barcodes": len(ordered),
                "barcodes": "|".join(ordered), "selected": ordered[0],
            })
    return pd.DataFrame(records)


def collapse_matrix_to_primary_patients(matrix, samples_are_rows, layer_name):
    df = matrix.copy() if samples_are_rows else matrix.T.copy()
    if df.index.duplicated().any():
        duplicated = df.index[df.index.duplicated()].unique().tolist()
        raise RuntimeError(
            f"[{layer_name}] barcodes exactement dupliqués avant sélection : {duplicated[:10]}"
        )
    all_ids = df.index.astype(str).tolist()
    n_raw_barcodes = len(all_ids)
    duplicate_audit = audit_duplicate_primary_samples(all_ids, layer_name)
    if not duplicate_audit.empty:
        path = OUT_DIR / f"duplicate_primary_samples_{layer_name}.csv"
        duplicate_audit.to_csv(path, index=False)
        log(f"    [{layer_name}] {len(duplicate_audit)} patients avec plusieurs "
            f"barcodes primaires -> {path.name}")
    selected = select_primary_tumor_samples(all_ids)
    if not selected:
        raise ValueError(f"[{layer_name}] aucun échantillon tumoral primaire code 01 détecté.")
    rows, patient_ids = [], []
    for patient_id, barcode in selected.items():
        row = df.loc[barcode]
        if isinstance(row, pd.DataFrame):
            raise RuntimeError(f"[{layer_name}] df.loc[{barcode!r}] retourne plusieurs lignes.")
        rows.append(row)
        patient_ids.append(patient_id)
    out = pd.DataFrame(rows, index=patient_ids)
    out.index.name = "PATIENT_ID"
    CROSS_LAYER_SELECTIONS[layer_name] = selected
    log(f"    [{layer_name}] barcodes bruts={n_raw_barcodes} -> "
        f"patients primaires uniques={len(out)}")
    return out.sort_index()


def read_omics_matrix_gz(path: Path, layer_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t", index_col=0, compression="gzip")
    return collapse_matrix_to_primary_patients(df, samples_are_rows=False, layer_name=layer_name)


def audit_cross_layer_aliquot_concordance(selected_by_layer):
    all_patients = set()
    for selections in selected_by_layer.values():
        all_patients.update(selections.keys())
    records = []
    for patient_id in sorted(all_patients):
        row = {"patient_id": patient_id}
        suffixes = {}
        for layer in LAYERS:
            barcode = selected_by_layer.get(layer, {}).get(patient_id)
            suffix = tcga_sample_suffix(barcode)
            row[f"{layer}_suffix"] = suffix
            if suffix is not None:
                suffixes[layer] = suffix
        unique_suffixes = set(suffixes.values())
        n_layers_present = len(suffixes)
        row["n_layers_present"] = n_layers_present
        row["evaluable"] = n_layers_present >= 2
        row["concordant"] = n_layers_present >= 2 and len(unique_suffixes) == 1
        row["four_layer_evaluable"] = n_layers_present == len(LAYERS)
        row["four_layer_concordant"] = (
            n_layers_present == len(LAYERS) and len(unique_suffixes) == 1
        )
        records.append(row)
    return pd.DataFrame(records)


def strip_ensembl_version(gene_id: str) -> str:
    return str(gene_id).split(".")[0]


def audit_mrna_scale(df: pd.DataFrame) -> Dict[str, float]:
    values = df.to_numpy(dtype=np.float64)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        raise RuntimeError("Aucune valeur mRNA finie détectée.")
    return {
        "min": float(np.min(finite)), "median": float(np.median(finite)),
        "p95": float(np.percentile(finite, 95)), "p99": float(np.percentile(finite, 99)),
        "max": float(np.max(finite)), "fraction_zero": float(np.mean(finite == 0)),
    }


def build_mrna_from_xena(xena_path: Path, mapping_path: Path):
    if CONFIG["mrna_apply_log_transform"]:
        raise RuntimeError(
            "CONFIG['mrna_apply_log_transform'] doit rester False : "
            "la matrice Xena est déjà en log2(TPM + 1)."
        )
    log("  [mrna] reconstruction depuis la matrice UCSC Xena STAR-TPM...")
    raw = pd.read_csv(xena_path, sep="\t", index_col=0, compression="gzip").T
    primary = collapse_matrix_to_primary_patients(raw, samples_are_rows=True, layer_name="mrna")
    if not mapping_path.exists():
        raise FileNotFoundError(f"Mapping Ensembl->HUGO introuvable : {mapping_path}")
    with open(mapping_path, encoding="utf-8") as f:
        id_to_symbol = json.load(f)
    stripped_cols = [strip_ensembl_version(c) for c in primary.columns]
    mapped_cols = [id_to_symbol.get(g) for g in stripped_cols]
    n_before = primary.shape[1]
    keep_mask = [s is not None for s in mapped_cols]
    primary = primary.loc[:, keep_mask]
    primary.columns = [s for s in mapped_cols if s is not None]
    n_after_mapping = primary.shape[1]
    primary = primary.T.groupby(level=0).mean().T
    log(f"    mapping HUGO : {n_before} -> {n_after_mapping} colonnes mappées "
        f"-> {primary.shape[1]} symboles uniques")
    pam50_missing = sorted(set(PAM50_GENES) - set(primary.columns))
    if pam50_missing:
        raise RuntimeError(f"Gènes PAM50 manquants après reconstruction mRNA : {pam50_missing}")
    log("    contrôle PAM50 : 50/50 gènes présents")
    scale_audit = audit_mrna_scale(primary)
    log(f"    échelle déclarée : {CONFIG['mrna_scale']}")
    log(f"    audit descriptif mRNA : {scale_audit}")
    return primary.sort_index(), scale_audit


MRNA_XENA_PATH = DATA_DIR / "TCGA-BRCA.star_tpm.tsv.gz"
MRNA_REBUILT_PATH = DATA_DIR / "mrna_hugo_mapped_primary_only.parquet"
MRNA_MANIFEST_PATH = DATA_DIR / "mrna_hugo_mapped_primary_only.manifest.json"
CNV_PATH = DATA_DIR / "Gistic2_CopyNumber_Gistic2_all_data_by_genes.gz"
RPPA_PATH = DATA_DIR / "RPPA_RBN.gz"
ARCHIVE_PATH = DATA_DIR / "brca_tcga_pan_can_atlas_2018.tar.gz"
MAPPING_PATH = DATA_DIR / "ensembl_to_hugo_mapping.json"

for name, path in {
    "mrna_xena_star_tpm": MRNA_XENA_PATH, "cnv": CNV_PATH, "rppa": RPPA_PATH,
    "mutations_clinical_archive": ARCHIVE_PATH, "ensembl_to_hugo_mapping": MAPPING_PATH,
}.items():
    register_input(name, path)

RAW: Dict[str, pd.DataFrame] = {}
MRNA_SCALE_AUDIT: Dict[str, float] = {}

log("=" * 80)
log(f"RUN_ID : {RUN_ID}  (dossier : {RUN_DIR})")
log(f"dim_mode : {CONFIG['dim_mode']}  |  dim_k : {CONFIG['dim_k']}")
log("=" * 80)


def current_mrna_manifest() -> Dict[str, Any]:
    return {
        "xena_sha256": sha256_file(MRNA_XENA_PATH),
        "mapping_sha256": sha256_file(MAPPING_PATH),
        "allowed_sample_type_codes": sorted(CONFIG["allowed_sample_type_codes"]),
        "duplicate_rule": CONFIG["duplicate_rule"],
        "mrna_scale": CONFIG["mrna_scale"],
        "mrna_apply_log_transform": CONFIG["mrna_apply_log_transform"],
    }


mrna_manifest_valid = False
if MRNA_REBUILT_PATH.exists() and MRNA_MANIFEST_PATH.exists():
    with open(MRNA_MANIFEST_PATH, encoding="utf-8") as f:
        saved_manifest = json.load(f)
    current_manifest = current_mrna_manifest()
    keys = ["xena_sha256", "mapping_sha256", "allowed_sample_type_codes",
            "duplicate_rule", "mrna_scale", "mrna_apply_log_transform"]
    mrna_manifest_valid = all(saved_manifest.get(k) == current_manifest.get(k) for k in keys)
    if not mrna_manifest_valid:
        log("  [mrna] cache trouvé mais manifeste invalide : reconstruction forcée.")

if MRNA_REBUILT_PATH.exists() and mrna_manifest_valid:
    log(f"  [mrna] chargé depuis le cache validé (partagé avec k100) : {MRNA_REBUILT_PATH.name}")
    RAW["mrna"] = pd.read_parquet(MRNA_REBUILT_PATH)
    RAW["mrna"].index = RAW["mrna"].index.astype(str)
    pam50_missing = sorted(set(PAM50_GENES) - set(RAW["mrna"].columns))
    if pam50_missing:
        raise RuntimeError(f"Gènes PAM50 manquants dans le cache mRNA : {pam50_missing}")
    log("    contrôle PAM50 du cache : 50/50 gènes présents")
    MRNA_SCALE_AUDIT = audit_mrna_scale(RAW["mrna"])
    log(f"    audit descriptif mRNA : {MRNA_SCALE_AUDIT}")
    with open(MRNA_MANIFEST_PATH, encoding="utf-8") as f:
        saved_manifest_full = json.load(f)
    CROSS_LAYER_SELECTIONS["mrna"] = saved_manifest_full.get("selected_samples", {})
else:
    RAW["mrna"], MRNA_SCALE_AUDIT = build_mrna_from_xena(MRNA_XENA_PATH, MAPPING_PATH)
    RAW["mrna"].to_parquet(MRNA_REBUILT_PATH)
    manifest = current_mrna_manifest()
    manifest["created_at"] = datetime.now().isoformat(timespec="seconds")
    manifest["scale_audit"] = MRNA_SCALE_AUDIT
    manifest["selected_samples"] = CROSS_LAYER_SELECTIONS.get("mrna", {})
    with open(MRNA_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    log(f"  [mrna] cache reconstruit sauvegardé : {MRNA_REBUILT_PATH.name}")

register_input("mrna_hugo_mapped_primary_only", MRNA_REBUILT_PATH)

log("  [cnv] lecture...")
RAW["cnv"] = read_omics_matrix_gz(CNV_PATH, layer_name="cnv")
log("  [rppa] lecture...")
RAW["rppa"] = read_omics_matrix_gz(RPPA_PATH, layer_name="rppa")

log("  [mutations] lecture depuis l'archive...")
with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
    maf_handle = tar.extractfile("brca_tcga_pan_can_atlas_2018/data_mutations.txt")
    clinical_handle = tar.extractfile("brca_tcga_pan_can_atlas_2018/data_clinical_patient.txt")
    if maf_handle is None or clinical_handle is None:
        raise FileNotFoundError("Fichiers mutation/clinique absents de l'archive.")
    maf = pd.read_csv(maf_handle, sep="\t", comment="#", low_memory=False)
    clinical = pd.read_csv(clinical_handle, sep="\t", comment="#", low_memory=False)

required_maf_columns = {"Tumor_Sample_Barcode", "Hugo_Symbol", "Variant_Classification"}
if not required_maf_columns.issubset(maf.columns):
    raise KeyError(f"Colonnes MAF manquantes : {required_maf_columns - set(maf.columns)}")

n_maf_before = len(maf)
maf = maf[maf["Variant_Classification"].isin(NONSYNONYMOUS)].copy()
all_mutation_barcodes = maf["Tumor_Sample_Barcode"].astype(str).unique().tolist()
duplicate_mutations = audit_duplicate_primary_samples(all_mutation_barcodes, layer="mutations")
if not duplicate_mutations.empty:
    path = OUT_DIR / "duplicate_primary_samples_mutations.csv"
    duplicate_mutations.to_csv(path, index=False)
    log(f"    [mutations] {len(duplicate_mutations)} patients avec plusieurs "
        f"barcodes primaires -> {path.name}")
selected_mutation_samples = select_primary_tumor_samples(all_mutation_barcodes)
CROSS_LAYER_SELECTIONS["mutations"] = selected_mutation_samples
selected_mutation_barcodes = set(selected_mutation_samples.values())
maf = maf[maf["Tumor_Sample_Barcode"].astype(str).isin(selected_mutation_barcodes)].copy()
maf["pid"] = maf["Tumor_Sample_Barcode"].map(tcga_patient_id)
log(f"    MAF : {n_maf_before} variantes -> {len(maf)} variantes non synonymes, "
    f"un barcode primaire par patient ({len(selected_mutation_barcodes)} patients)")

RAW["mutations"] = (
    maf.groupby(["pid", "Hugo_Symbol"]).size().unstack(fill_value=0)
    .clip(upper=1).astype(np.int8).sort_index()
)

if "SUBTYPE" not in clinical.columns:
    raise KeyError("Colonne SUBTYPE absente du fichier clinique.")
if "PATIENT_ID" not in clinical.columns:
    raise KeyError("Colonne PATIENT_ID absente du fichier clinique.")

labels_all = (
    clinical.set_index("PATIENT_ID")["SUBTYPE"].dropna().astype(str)
    .str.replace(r"^BRCA_", "", regex=True)
)
labels_all = labels_all[labels_all.isin(["LumA", "LumB", "Her2", "Basal", "Normal"])]
labels_all.index = labels_all.index.map(tcga_patient_id)
labels_all.name = "PAM50"

if labels_all.index.duplicated().any():
    duplicated_labels = labels_all.index[labels_all.index.duplicated(keep=False)].unique().tolist()
    consistency = labels_all.groupby(level=0).nunique()
    conflicting = consistency[consistency > 1].index.tolist()
    if conflicting:
        raise RuntimeError(f"Labels PAM50 contradictoires pour certains patients : {conflicting[:10]}")
    labels_all = labels_all[~labels_all.index.duplicated(keep="first")]
    log(f"Labels cliniques dupliqués mais cohérents : {len(duplicated_labels)} patients dédupliqués.")

for layer, df in RAW.items():
    if df.index.duplicated().any():
        duplicated = df.index[df.index.duplicated()].unique().tolist()
        raise RuntimeError(f"Patients dupliqués dans {layer} : {duplicated[:10]}")

common = set(labels_all.index)
for layer in LAYERS:
    common &= set(RAW[layer].index)
common = sorted(common)
if not common:
    raise ValueError("Intersection vide entre les quatre omiques et PAM50.")

RAW = {layer: RAW[layer].loc[common].copy() for layer in LAYERS}
labels = labels_all.loc[common].copy()
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels.to_numpy())
N = len(y)
class_counts = dict(zip(label_encoder.classes_, np.bincount(y).tolist()))

log("")
log(f"Cohorte finale : {N} patients")
log(f"Classes PAM50 : {class_counts}")
for layer in LAYERS:
    log(f"{SHORT[layer]:<12}: {RAW[layer].shape}")
if np.bincount(y).min() < int(CONFIG["cv_folds"]):
    raise ValueError("Une classe contient moins de patients que le nombre de folds.")

if set(CROSS_LAYER_SELECTIONS.keys()) == set(LAYERS):
    concordance_df = audit_cross_layer_aliquot_concordance(CROSS_LAYER_SELECTIONS)
else:
    concordance_df = pd.DataFrame()

NF = int(CONFIG["cv_folds"])
NR = int(CONFIG["cv_repeats"])
splitter = RepeatedStratifiedKFold(n_splits=NF, n_repeats=NR, random_state=SEED)
SPLITS: List[Tuple[np.ndarray, np.ndarray]] = list(splitter.split(np.zeros((N, 1)), y))
REPEAT_OF = [i // NF for i in range(len(SPLITS))]
FOLD_OF = [i % NF for i in range(len(SPLITS))]
log(f"{len(SPLITS)} splits = {NR} répétitions x {NF} folds")


def score_features(df: pd.DataFrame, stat: str) -> pd.Series:
    if stat == "mad":
        medians = df.median(axis=0)
        return (df - medians).abs().median(axis=0)
    if stat == "iqr":
        quantiles = df.quantile([0.25, 0.75], axis=0)
        return quantiles.loc[0.75] - quantiles.loc[0.25]
    if stat == "var":
        return df.var(axis=0)
    raise ValueError(f"Statistique de dispersion inconnue : {stat}")


def fit_reducer(X_train: np.ndarray, layer: str, seed: int):
    max_allowed = min(X_train.shape[0] - 1, X_train.shape[1] - 1)
    if max_allowed < 1:
        raise ValueError(f"Réduction impossible pour {layer}, shape={X_train.shape}")
    Reducer = TruncatedSVD if layer in BINARY_LAYERS else PCA
    if CONFIG["dim_mode"] == "fixed_k":
        n_components = min(int(CONFIG["dim_k"]), max_allowed)
        reducer = Reducer(n_components=n_components, random_state=seed)
        reducer.fit(X_train)
        return reducer, n_components, float(reducer.explained_variance_ratio_.sum())
    if CONFIG["dim_mode"] == "fixed_variance":
        k_max = min(int(CONFIG["dim_k_max"]), max_allowed)
        probe = Reducer(n_components=k_max, random_state=seed)
        probe.fit(X_train)
        cumulative_variance = np.cumsum(probe.explained_variance_ratio_)
        target = float(CONFIG["dim_variance_target"])
        if cumulative_variance[-1] >= target:
            n_components = int(np.searchsorted(cumulative_variance, target) + 1)
        else:
            n_components = k_max
        reducer = Reducer(n_components=n_components, random_state=seed)
        reducer.fit(X_train)
        return reducer, n_components, float(reducer.explained_variance_ratio_.sum())
    raise ValueError(f"Mode de réduction inconnu : {CONFIG['dim_mode']}")


def preprocess_layer(layer: str, train_index: np.ndarray, test_index: np.ndarray, seed: int) -> Dict[str, Any]:
    df = RAW[layer]
    train_df = df.iloc[train_index].copy()
    test_df = df.iloc[test_index].copy()
    metadata: Dict[str, Any] = {
        "layer": layer, "n_features_initial": int(train_df.shape[1]),
        "n_train": int(len(train_index)), "n_test": int(len(test_index)),
    }

    if layer == "rppa":
        missing_rate = train_df.isna().mean(axis=0)
        keep_columns = missing_rate[missing_rate <= float(CONFIG["rppa_max_missing"])].index
        train_df = train_df.loc[:, keep_columns]
        test_df = test_df.loc[:, keep_columns]
        if CONFIG["rppa_imputer"] == "knn":
            imputer = KNNImputer(n_neighbors=int(CONFIG["rppa_knn_k"]))
        else:
            imputer = SimpleImputer(strategy="median")
        n_columns_before = train_df.shape[1]
        train_array = imputer.fit_transform(train_df)
        test_array = imputer.transform(test_df)
        if train_array.shape[1] != n_columns_before:
            raise RuntimeError("[rppa] l'imputation a modifié le nombre de variables.")
        train_df = pd.DataFrame(train_array, index=train_df.index, columns=train_df.columns)
        test_df = pd.DataFrame(test_array, index=test_df.index, columns=test_df.columns)
    elif train_df.isna().any().any() or test_df.isna().any().any():
        imputer = SimpleImputer(strategy="median")
        train_df = pd.DataFrame(imputer.fit_transform(train_df), index=train_df.index, columns=train_df.columns)
        test_df = pd.DataFrame(imputer.transform(test_df), index=test_df.index, columns=test_df.columns)

    if layer == "mutations":
        mutation_frequency = train_df.mean(axis=0)
        keep_columns = mutation_frequency[
            (mutation_frequency >= float(CONFIG["mut_freq_lo"]))
            & (mutation_frequency <= float(CONFIG["mut_freq_hi"]))
        ].index
        train_df = train_df.loc[:, keep_columns]
        test_df = test_df.loc[:, keep_columns]

    variances = train_df.var(axis=0)
    keep_columns = variances[variances > float(CONFIG["min_variance"])].index
    train_df = train_df.loc[:, keep_columns]
    test_df = test_df.loc[:, keep_columns]

    if layer in {"mrna", "cnv"} and train_df.shape[1] > int(CONFIG["filter_top_k"]):
        feature_scores = score_features(train_df, str(CONFIG["filter_stat"]))
        selected_columns = feature_scores.nlargest(int(CONFIG["filter_top_k"])).index
        train_df = train_df.loc[:, selected_columns]
        test_df = test_df.loc[:, selected_columns]

    if train_df.shape[1] < 2:
        raise ValueError(f"{layer}: moins de deux variables après filtrage.")

    metadata["n_features_selected"] = int(train_df.shape[1])
    metadata["selected_features"] = train_df.columns.astype(str).tolist()
    X_train = train_df.to_numpy(dtype=np.float64, copy=True)
    X_test = test_df.to_numpy(dtype=np.float64, copy=True)
    assert_finite_matrix(X_train, f"{layer}/train")
    assert_finite_matrix(X_test, f"{layer}/test")

    if layer not in BINARY_LAYERS:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        metadata["standardized"] = True
    else:
        metadata["standardized"] = False

    reducer, n_components, retained_variance = fit_reducer(X_train, layer=layer, seed=seed)
    Z_train = reducer.transform(X_train).astype(np.float32)
    Z_test = reducer.transform(X_test).astype(np.float32)
    assert_finite_matrix(Z_train, f"{layer}/train réduit")
    assert_finite_matrix(Z_test, f"{layer}/test réduit")

    metadata["n_components"] = int(n_components)
    metadata["retained_variance"] = float(retained_variance)
    metadata["dim_mode"] = CONFIG["dim_mode"]
    metadata["filter_stat"] = CONFIG["filter_stat"]

    return {"X_train": Z_train, "X_test": Z_test, "metadata": metadata}


cache = ckpt_load("preprocessing_cache")
if cache is None:
    cache = {"representations": {}, "metadata": {}}
REPRESENTATIONS = cache["representations"]
PREPROCESS_METADATA = cache["metadata"]

for split_index, (train_index, test_index) in enumerate(SPLITS):
    split_complete = all((split_index, layer) in REPRESENTATIONS for layer in LAYERS)
    if split_complete:
        continue
    log(f"Split {split_index + 1}/{len(SPLITS)} (recalcul -- ne devrait pas arriver si le cache est intact)")
    for layer in LAYERS:
        result = preprocess_layer(
            layer=layer, train_index=train_index, test_index=test_index,
            seed=SEED + REPEAT_OF[split_index],
        )
        REPRESENTATIONS[(split_index, layer)] = (result["X_train"], result["X_test"])
        PREPROCESS_METADATA[(split_index, layer)] = result["metadata"]
    if (split_index + 1) % int(CONFIG["cache_save_every"]) == 0 or split_index == len(SPLITS) - 1:
        ckpt_save("preprocessing_cache", {"representations": REPRESENTATIONS, "metadata": PREPROCESS_METADATA})

log("")
log(f"Cache prêt : {len(REPRESENTATIONS)}/{len(SPLITS) * len(LAYERS)} entrées "
    f"({N} patients, dim_k={CONFIG['dim_k']})")
print("\n Contexte pca50 rechargé, prêt pour l'évaluation.")

#3.Evalution du pca sur 50

In [ ]:
"""
Évaluation 15 panels x 3 classifieurs -- pca50 (RUN_ID=2026-07-26, dim_k=50)
"""

import os
import json
import math
import time
import warnings
from pathlib import Path
from itertools import combinations
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# =============================================================================
# 0. VÉRIFICATION
# =============================================================================

REQUIRED_GLOBALS = [
    "CONFIG", "SEED", "LAYERS", "SHORT", "SPLITS", "REPEAT_OF", "FOLD_OF",
    "REPRESENTATIONS", "y", "N", "OUT_DIR", "RUN_ID",
]
missing_globals = [name for name in REQUIRED_GLOBALS if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Exécutez d'abord la cellule 2 (rechargement pca50) dans la même session. "
        f"Variables manquantes : {missing_globals}"
    )

if RUN_ID != "2026-07-26":
    raise RuntimeError(
        f"RUN_ID courant = '{RUN_ID}', attendu '2026-07-26'."
    )

if int(CONFIG["dim_k"]) != 50:
    raise RuntimeError(
        f"CONFIG['dim_k'] = {CONFIG['dim_k']}, attendu 50."
    )

OUT_DIR = Path(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR = OUT_DIR / "model_evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUN_ID confirmé : {RUN_ID}")
print(f"dim_k confirmé : {CONFIG['dim_k']}")
print(f"Résultats d'évaluation écrits dans : {EVAL_DIR}")

EVAL_CONFIG: Dict[str, Any] = {
    "primary_metric": "balanced_accuracy",
    "bootstrap_replicates_repeat_level": 10000,
    "fdr_alpha": 0.05,
    "minimal_panel_delta": 0.03,
    "random_forest": {
        "n_estimators": 500, "max_features": "sqrt",
        "min_samples_leaf": 2, "n_jobs": -1,
    },
    "xgboost": {
        "n_estimators": 500, "max_depth": 4, "learning_rate": 0.03,
        "subsample": 0.85, "colsample_bytree": 0.85,
        "reg_lambda": 1.0, "reg_alpha": 0.0, "min_child_weight": 1.0, "n_jobs": -1,
    },
    "svm": {
        "C": 1.0, "kernel": "rbf", "gamma": "scale",
        "class_weight": "balanced", "probability": False,
    },
}


def eval_log(message: str) -> None:
    print(message)


# =============================================================================
# 1. PANELS
# =============================================================================

def canonical_panel(panel):
    order = {layer: i for i, layer in enumerate(LAYERS)}
    return tuple(sorted(panel, key=lambda x: order[x]))


PANELS = []
for size in range(1, len(LAYERS) + 1):
    PANELS.extend(canonical_panel(tuple(p)) for p in combinations(LAYERS, size))
FULL_PANEL = tuple(LAYERS)
if len(PANELS) != 15:
    raise RuntimeError(f"15 panels attendus, obtenu : {len(PANELS)}")


def panel_key(panel):
    return "+".join(panel)


def panel_label(panel):
    return "+".join(SHORT[layer] for layer in panel)


# =============================================================================
# 2. MATRICES PAR PANEL
# =============================================================================

def get_panel_matrices(split_index, panel):
    train_blocks, test_blocks = [], []
    for layer in panel:
        key = (split_index, layer)
        if key not in REPRESENTATIONS:
            raise KeyError(f"Représentation absente : {key}")
        Xtr, Xte = REPRESENTATIONS[key]
        train_blocks.append(Xtr)
        test_blocks.append(Xte)
    X_train = np.concatenate(train_blocks, axis=1)
    X_test = np.concatenate(test_blocks, axis=1)
    if not np.isfinite(X_train).all() or not np.isfinite(X_test).all():
        raise RuntimeError(f"Valeurs non finies : panel {panel_key(panel)}, split {split_index}.")
    return X_train, X_test


# =============================================================================
# 3. CLASSIFIEURS
# =============================================================================

def make_classifier(classifier_name, seed, n_classes):
    if classifier_name == "RF":
        return RandomForestClassifier(random_state=seed, **EVAL_CONFIG["random_forest"])
    if classifier_name == "XGB":
        return XGBClassifier(
            objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss",
            random_state=seed, verbosity=0, **EVAL_CONFIG["xgboost"],
        )
    if classifier_name == "SVM":
        return SVC(random_state=seed, **EVAL_CONFIG["svm"])
    if classifier_name == "DUMMY":
        return DummyClassifier(strategy="prior", random_state=seed)
    raise ValueError(f"Classifieur inconnu : {classifier_name}")


CLASSIFIERS = ["RF", "XGB", "SVM"]


# =============================================================================
# 4. ÉVALUATION -- REPRISE FINE
# =============================================================================

def compute_metrics(y_true, y_pred):
    return {
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }


fold_results_path = EVAL_DIR / "panel_fold_results.csv"

if fold_results_path.exists():
    fold_results_df = pd.read_csv(fold_results_path)
    eval_log(f"Rechargement partiel : {fold_results_path.name} "
             f"({len(fold_results_df)} lignes déjà présentes)")
else:
    fold_results_df = pd.DataFrame()

done_split_classifier = set()
if not fold_results_df.empty:
    counts = fold_results_df.groupby(["split", "classifier"]).size()
    expected_rows_per_split_classifier = len(PANELS) + 1
    for (split_index, classifier_name), n_rows in counts.items():
        if n_rows >= expected_rows_per_split_classifier:
            done_split_classifier.add((int(split_index), classifier_name))

rows = fold_results_df.to_dict("records") if not fold_results_df.empty else []
n_classes = int(len(np.unique(y)))
start_time = time.time()
n_total = len(SPLITS) * len(CLASSIFIERS)
n_done_at_start = len(done_split_classifier)
eval_log(f"Reprise : {n_done_at_start}/{n_total} (split, classifier) déjà complets")

for split_index, (train_index, test_index) in enumerate(SPLITS):
    repeat_index = int(REPEAT_OF[split_index])
    fold_index = int(FOLD_OF[split_index])
    classifier_seed = int(SEED + repeat_index)
    y_train = y[train_index]
    y_test = y[test_index]

    any_new_this_split = False

    for classifier_name in CLASSIFIERS:
        if (split_index, classifier_name) in done_split_classifier:
            continue
        any_new_this_split = True

        dummy = make_classifier("DUMMY", seed=classifier_seed, n_classes=n_classes)
        dummy.fit(np.zeros((len(train_index), 1), dtype=np.float32), y_train)
        dummy_pred = dummy.predict(np.zeros((len(test_index), 1), dtype=np.float32))
        dummy_metrics = compute_metrics(y_test, dummy_pred)
        rows.append({
            "classifier": classifier_name, "split": split_index, "repeat": repeat_index,
            "fold": fold_index, "panel": "EMPTY", "panel_label": "Baseline", "panel_size": 0,
            "balanced_accuracy": dummy_metrics["balanced_accuracy"],
            "macro_f1": dummy_metrics["macro_f1"], "accuracy": dummy_metrics["accuracy"],
            "fit_seconds": 0.0,
        })

        sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)

        for panel in PANELS:
            X_train, X_test = get_panel_matrices(split_index, panel)
            model = make_classifier(classifier_name, seed=classifier_seed, n_classes=n_classes)

            fit_start = time.time()
            if classifier_name in {"RF", "XGB"}:
                model.fit(X_train, y_train, sample_weight=sample_weight)
            else:
                model.fit(X_train, y_train)
            fit_seconds = time.time() - fit_start

            predictions = model.predict(X_test)
            metrics = compute_metrics(y_test, predictions)
            rows.append({
                "classifier": classifier_name, "split": split_index, "repeat": repeat_index,
                "fold": fold_index, "panel": panel_key(panel), "panel_label": panel_label(panel),
                "panel_size": len(panel), "balanced_accuracy": metrics["balanced_accuracy"],
                "macro_f1": metrics["macro_f1"], "accuracy": metrics["accuracy"],
                "fit_seconds": float(fit_seconds),
            })

        done_split_classifier.add((split_index, classifier_name))

    if any_new_this_split:
        tmp_path = fold_results_path.with_suffix(".csv.tmp")
        pd.DataFrame(rows).to_csv(tmp_path, index=False)
        os.replace(tmp_path, fold_results_path)
        eval_log(f"  progression : {len(done_split_classifier)}/{n_total} "
                 f"(split, classifier) -- ({time.time() - start_time:.0f}s cette session)")

fold_results = pd.DataFrame(rows)
eval_log(f"Résultats fold-level : {len(fold_results)} lignes "
         f"({len(done_split_classifier)}/{n_total} (split, classifier) couverts)")

if len(done_split_classifier) != n_total:
    raise RuntimeError(
        f"Évaluation incomplète : {len(done_split_classifier)}/{n_total}. "
        "Relancez ce script pour terminer."
    )


# =============================================================================
# 5. AGRÉGATION
# =============================================================================

METRICS = ["balanced_accuracy", "macro_f1", "accuracy"]

repeat_results = (
    fold_results.groupby(["classifier", "repeat", "panel", "panel_label", "panel_size"], as_index=False)[METRICS]
    .mean()
)
repeat_results.to_csv(EVAL_DIR / "panel_repeat_results.csv", index=False)

panel_summary = (
    repeat_results.groupby(["classifier", "panel", "panel_label", "panel_size"], as_index=False)[METRICS]
    .agg(["mean", "std"])
)
panel_summary.columns = ["_".join(str(x) for x in c if str(x) != "") for c in panel_summary.columns.to_flat_index()]
panel_summary.to_csv(EVAL_DIR / "panel_summary.csv", index=False)


# =============================================================================
# 6. OUTILS STATISTIQUES
# =============================================================================

def bootstrap_mean_ci(values, n_boot, seed, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    draws = rng.choice(values, size=(n_boot, values.size), replace=True)
    means = draws.mean(axis=1)
    return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))


def exact_sign_flip_pvalue_greater(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return float("nan")
    observed = float(values.mean())
    n = values.size
    permuted_means = []
    for mask in range(1 << n):
        signs = np.ones(n, dtype=float)
        for i in range(n):
            if (mask >> i) & 1:
                signs[i] = -1.0
        permuted_means.append(float(np.mean(values * signs)))
    permuted_means = np.asarray(permuted_means)
    return float((np.sum(permuted_means >= observed) + 1) / (len(permuted_means) + 1))


# =============================================================================
# 7. GAIN D'INTÉGRATION
# =============================================================================

primary_metric = str(EVAL_CONFIG["primary_metric"])

repeat_lookup = {
    (row.classifier, int(row.repeat), row.panel): float(getattr(row, primary_metric))
    for row in repeat_results.itertuples(index=False)
}

gain_rows = []
for classifier_name in CLASSIFIERS:
    for repeat_index in sorted(repeat_results["repeat"].unique()):
        for panel in PANELS:
            if len(panel) < 2:
                continue
            panel_name = panel_key(panel)
            panel_score = repeat_lookup[(classifier_name, int(repeat_index), panel_name)]
            constituent_scores = {
                layer: repeat_lookup[(classifier_name, int(repeat_index), panel_key((layer,)))]
                for layer in panel
            }
            best_single_layer = max(constituent_scores, key=constituent_scores.get)
            best_single_score = constituent_scores[best_single_layer]
            gain = panel_score - best_single_score
            gain_rows.append({
                "classifier": classifier_name, "repeat": int(repeat_index), "panel": panel_name,
                "panel_label": panel_label(panel), "panel_size": len(panel),
                "panel_score": float(panel_score), "best_constituent_single": best_single_layer,
                "best_constituent_single_score": float(best_single_score),
                "integration_gain": float(gain), "gain_positive": bool(gain > 0),
            })

integration_gains = pd.DataFrame(gain_rows)
integration_gains.to_csv(EVAL_DIR / "integration_gain_repeat_level.csv", index=False)

gain_summary_rows = []
for (classifier_name, panel_name, panel_display, panel_size), group in integration_gains.groupby(
    ["classifier", "panel", "panel_label", "panel_size"]
):
    values = group["integration_gain"].to_numpy(dtype=float)
    ci_low, ci_high = bootstrap_mean_ci(
        values, n_boot=int(EVAL_CONFIG["bootstrap_replicates_repeat_level"]),
        seed=SEED + sum(ord(c) for c in classifier_name + panel_name),
    )
    p_value = exact_sign_flip_pvalue_greater(values)
    gain_summary_rows.append({
        "classifier": classifier_name, "panel": panel_name, "panel_label": panel_display,
        "panel_size": int(panel_size), "mean_gain": float(np.mean(values)),
        "sd_gain": float(np.std(values, ddof=1)), "ci95_low": ci_low, "ci95_high": ci_high,
        "positive_gain_frequency": float(np.mean(values > 0)), "n_repeats": int(len(values)),
        "p_signflip_greater": p_value,
    })

gain_summary = pd.DataFrame(gain_summary_rows)
gain_summary["q_bh"] = np.nan
gain_summary["significant_positive_gain"] = False
for classifier_name, indices in gain_summary.groupby("classifier").groups.items():
    p_values = gain_summary.loc[indices, "p_signflip_greater"].to_numpy(dtype=float)
    reject, q_values, _, _ = multipletests(p_values, alpha=float(EVAL_CONFIG["fdr_alpha"]), method="fdr_bh")
    gain_summary.loc[indices, "q_bh"] = q_values
    gain_summary.loc[indices, "significant_positive_gain"] = (
        reject & (gain_summary.loc[indices, "mean_gain"].to_numpy(dtype=float) > 0)
    )
gain_summary.to_csv(EVAL_DIR / "integration_gain_summary.csv", index=False)


# =============================================================================
# 8. SHAPLEY
# =============================================================================

def all_subsets_without(items, excluded):
    remaining = tuple(item for item in items if item != excluded)
    subsets = [tuple()]
    for size in range(1, len(remaining) + 1):
        subsets.extend(canonical_panel(tuple(s)) for s in combinations(remaining, size))
    return subsets


def exact_performance_shapley(value_function):
    n = len(LAYERS)
    shapley = {layer: 0.0 for layer in LAYERS}
    for layer in LAYERS:
        for subset in all_subsets_without(tuple(LAYERS), layer):
            subset = canonical_panel(subset)
            with_layer = canonical_panel(subset + (layer,))
            s = len(subset)
            weight = math.factorial(s) * math.factorial(n - s - 1) / math.factorial(n)
            shapley[layer] += weight * (value_function[with_layer] - value_function[subset])
    return shapley


shapley_repeat_rows = []
for classifier_name in CLASSIFIERS:
    for repeat_index in sorted(repeat_results["repeat"].unique()):
        value_function = {}
        value_function[tuple()] = repeat_lookup[(classifier_name, int(repeat_index), "EMPTY")]
        for panel in PANELS:
            value_function[panel] = repeat_lookup[(classifier_name, int(repeat_index), panel_key(panel))]
        shapley_values = exact_performance_shapley(value_function)
        total_shapley = sum(shapley_values.values())
        total_gain = value_function[FULL_PANEL] - value_function[tuple()]
        if not np.isclose(total_shapley, total_gain, atol=1e-8):
            raise RuntimeError(
                f"Échec propriété d'efficacité Shapley : {classifier_name}, repeat={repeat_index}."
            )
        for layer, value in shapley_values.items():
            shapley_repeat_rows.append({
                "classifier": classifier_name, "repeat": int(repeat_index), "layer": layer,
                "layer_label": SHORT[layer], "performance_shapley": float(value),
                "full_minus_baseline": float(total_gain),
            })

shapley_repeat = pd.DataFrame(shapley_repeat_rows)
shapley_repeat.to_csv(EVAL_DIR / "performance_shapley_repeat_level.csv", index=False)

shapley_summary_rows = []
for (classifier_name, layer, layer_label), group in shapley_repeat.groupby(["classifier", "layer", "layer_label"]):
    values = group["performance_shapley"].to_numpy(dtype=float)
    ci_low, ci_high = bootstrap_mean_ci(
        values, n_boot=int(EVAL_CONFIG["bootstrap_replicates_repeat_level"]),
        seed=SEED + sum(ord(c) for c in classifier_name + layer),
    )
    shapley_summary_rows.append({
        "classifier": classifier_name, "layer": layer, "layer_label": layer_label,
        "mean_performance_shapley": float(np.mean(values)), "sd_performance_shapley": float(np.std(values, ddof=1)),
        "ci95_low": ci_low, "ci95_high": ci_high, "positive_frequency": float(np.mean(values > 0)),
    })
shapley_summary = pd.DataFrame(shapley_summary_rows)
shapley_summary.to_csv(EVAL_DIR / "performance_shapley_summary.csv", index=False)


# =============================================================================
# 9. CLASSEMENT ET ROBUSTESSE
# =============================================================================

panel_mean_scores = (
    repeat_results[repeat_results["panel"] != "EMPTY"]
    .groupby(["classifier", "panel", "panel_label", "panel_size"], as_index=False)[primary_metric]
    .mean().rename(columns={primary_metric: "mean_score"})
)
panel_mean_scores["rank_within_classifier"] = (
    panel_mean_scores.groupby("classifier")["mean_score"].rank(method="average", ascending=False)
)
panel_mean_scores.to_csv(EVAL_DIR / "panel_classifier_ranking.csv", index=False)

rank_pivot = panel_mean_scores.pivot(index="panel", columns="classifier", values="rank_within_classifier")
rank_correlation_rows = []
for classifier_a, classifier_b in combinations(CLASSIFIERS, 2):
    rho, p_value = spearmanr(rank_pivot[classifier_a], rank_pivot[classifier_b])
    rank_correlation_rows.append({
        "classifier_a": classifier_a, "classifier_b": classifier_b,
        "spearman_rho": float(rho), "p_value": float(p_value),
    })
rank_correlations = pd.DataFrame(rank_correlation_rows)
rank_correlations.to_csv(EVAL_DIR / "classifier_rank_correlations.csv", index=False)

robust_panel_summary = (
    panel_mean_scores.groupby(["panel", "panel_label", "panel_size"], as_index=False)
    .agg(
        mean_score_across_classifiers=("mean_score", "mean"),
        sd_score_across_classifiers=("mean_score", "std"),
        mean_rank_across_classifiers=("rank_within_classifier", "mean"),
        worst_rank_across_classifiers=("rank_within_classifier", "max"),
        top3_frequency=("rank_within_classifier", lambda x: float(np.mean(x <= 3))),
    )
    .sort_values(["mean_rank_across_classifiers", "mean_score_across_classifiers"], ascending=[True, False])
)
robust_panel_summary.to_csv(EVAL_DIR / "panel_robustness_across_classifiers.csv", index=False)


# =============================================================================
# 10. PANEL MINIMAL
# =============================================================================

best_overall_score = float(robust_panel_summary["mean_score_across_classifiers"].max())
minimal_threshold = best_overall_score - float(EVAL_CONFIG["minimal_panel_delta"])
minimal_candidates = robust_panel_summary[
    robust_panel_summary["mean_score_across_classifiers"] >= minimal_threshold
].copy()
minimal_candidates = minimal_candidates.sort_values(
    ["panel_size", "mean_rank_across_classifiers", "mean_score_across_classifiers"],
    ascending=[True, True, False],
)
minimal_sufficient_panel_row = minimal_candidates.iloc[0] if not minimal_candidates.empty else None
minimal_candidates.to_csv(EVAL_DIR / "minimal_sufficient_panel_candidates.csv", index=False)

minimal_panel_per_classifier_rows = []
for classifier_name in CLASSIFIERS:
    subset = panel_mean_scores[panel_mean_scores["classifier"] == classifier_name]
    best_score_clf = float(subset["mean_score"].max())
    threshold_clf = best_score_clf - float(EVAL_CONFIG["minimal_panel_delta"])
    candidates_clf = subset[subset["mean_score"] >= threshold_clf].copy()
    candidates_clf["panel_size"] = candidates_clf["panel"].apply(lambda p: len(p.split("+")))
    candidates_clf = candidates_clf.sort_values(["panel_size", "mean_score"], ascending=[True, False])
    if not candidates_clf.empty:
        best_row = candidates_clf.iloc[0]
        minimal_panel_per_classifier_rows.append({
            "classifier": classifier_name, "panel": best_row["panel"],
            "panel_label": best_row["panel_label"], "panel_size": int(best_row["panel_size"]),
            "mean_score": float(best_row["mean_score"]),
        })

minimal_panel_per_classifier = pd.DataFrame(minimal_panel_per_classifier_rows)
minimal_panel_per_classifier.to_csv(EVAL_DIR / "minimal_sufficient_panel_per_classifier.csv", index=False)

panels_agree_across_classifiers = (
    minimal_panel_per_classifier["panel"].nunique() == 1
    if not minimal_panel_per_classifier.empty else None
)


# =============================================================================
# 11. SYNTHÈSE
# =============================================================================

def best_panel_of_size(size):
    subset = robust_panel_summary[robust_panel_summary["panel_size"] == size]
    if subset.empty:
        return None
    return subset.sort_values(
        ["mean_score_across_classifiers", "mean_rank_across_classifiers"], ascending=[False, True]
    ).iloc[0]


best_single = best_panel_of_size(1)
best_pair = best_panel_of_size(2)
best_triplet = best_panel_of_size(3)
best_overall = robust_panel_summary.iloc[0]

best_gain_row = gain_summary.sort_values(
    ["mean_gain", "positive_gain_frequency"], ascending=[False, False]
).iloc[0]

dominant_layer_row = (
    shapley_summary.groupby(["layer", "layer_label"], as_index=False)["mean_performance_shapley"]
    .mean().sort_values("mean_performance_shapley", ascending=False).iloc[0]
)

significant_gain_panels = gain_summary[gain_summary["significant_positive_gain"]].copy()
does_multiomics_improve = bool(not significant_gain_panels.empty)

primary_answers = {
    "run_id": RUN_ID,
    "dim_k": int(CONFIG["dim_k"]),
    "primary_metric": primary_metric,
    "does_multiomics_improve_prediction": {
        "answer": does_multiomics_improve,
        "n_significant_panel_classifier_results": int(len(significant_gain_panels)),
    },
    "best_single_omic": None if best_single is None else {
        "panel": str(best_single["panel"]), "panel_label": str(best_single["panel_label"]),
        "mean_score_across_classifiers": float(best_single["mean_score_across_classifiers"]),
        "mean_rank_across_classifiers": float(best_single["mean_rank_across_classifiers"]),
    },
    "best_pair": None if best_pair is None else {
        "panel": str(best_pair["panel"]), "panel_label": str(best_pair["panel_label"]),
        "mean_score_across_classifiers": float(best_pair["mean_score_across_classifiers"]),
        "mean_rank_across_classifiers": float(best_pair["mean_rank_across_classifiers"]),
    },
    "best_triplet": None if best_triplet is None else {
        "panel": str(best_triplet["panel"]), "panel_label": str(best_triplet["panel_label"]),
        "mean_score_across_classifiers": float(best_triplet["mean_score_across_classifiers"]),
        "mean_rank_across_classifiers": float(best_triplet["mean_rank_across_classifiers"]),
    },
    "best_overall_panel": {
        "panel": str(best_overall["panel"]), "panel_label": str(best_overall["panel_label"]),
        "panel_size": int(best_overall["panel_size"]),
        "mean_score_across_classifiers": float(best_overall["mean_score_across_classifiers"]),
        "mean_rank_across_classifiers": float(best_overall["mean_rank_across_classifiers"]),
    },
    "minimal_sufficient_panel_global": None if minimal_sufficient_panel_row is None else {
        "panel": str(minimal_sufficient_panel_row["panel"]),
        "panel_label": str(minimal_sufficient_panel_row["panel_label"]),
        "panel_size": int(minimal_sufficient_panel_row["panel_size"]),
        "mean_score_across_classifiers": float(minimal_sufficient_panel_row["mean_score_across_classifiers"]),
        "delta_from_best_threshold": float(EVAL_CONFIG["minimal_panel_delta"]),
    },
    "minimal_sufficient_panel_per_classifier": minimal_panel_per_classifier_rows,
    "minimal_panels_agree_across_classifiers": panels_agree_across_classifiers,
    "largest_incremental_gain": {
        "classifier": str(best_gain_row["classifier"]), "panel": str(best_gain_row["panel"]),
        "mean_gain": float(best_gain_row["mean_gain"]),
        "ci95_low": float(best_gain_row["ci95_low"]), "ci95_high": float(best_gain_row["ci95_high"]),
        "q_bh": float(best_gain_row["q_bh"]),
    },
    "most_dominant_layer_by_performance_shapley": {
        "layer": str(dominant_layer_row["layer"]), "layer_label": str(dominant_layer_row["layer_label"]),
        "mean_performance_shapley_across_classifiers": float(dominant_layer_row["mean_performance_shapley"]),
    },
}

with open(EVAL_DIR / "primary_answers.json", "w", encoding="utf-8") as f:
    json.dump(primary_answers, f, indent=2, ensure_ascii=False)

eval_log("")
eval_log("=" * 80)
eval_log(f"ÉVALUATION TERMINÉE (RUN_ID={RUN_ID}, dim_k={CONFIG['dim_k']})")
eval_log("=" * 80)
eval_log(f"Panel minimal (global) : {primary_answers['minimal_sufficient_panel_global']}")
eval_log(f"Accord inter-classifieur : {primary_answers['minimal_panels_agree_across_classifiers']}")